# A frozen-encoder router

**Part 3 of "LLMs the Hard Way"** (branch `frozen-router`). `01_modernbert_by_hand.ipynb` built ModernBERT's forward pass by hand and showed that a fixed classification head can't route to agents that change. This notebook turns the **frozen** encoder into a **multi-agent router**: for each candidate agent, read (conversation + agent description) in one pass, and let a small trained head score how well they fit.

```text
chat history + current agent + Agent A  →  encoder  →  head  →  score A
chat history + current agent + Agent B  →  encoder  →  head  →  score B      →  softmax  →  chosen agent
...
```

**What it covers.** The cell numbers are those of the original single notebook, so they start at 34:

| Phase | Cells | Topic |
|---|---|---|
| 11 | 34–40 | The routing problem, two encoders (ModernBERT-large or Nemotron-1B), the cross-encoder input, a compatibility head, scoring 5 agents |
| 12 | 41–46 | Training: positive and negative pairs, listwise loss, frozen features, leave-domains-out selection; a separate **choice head** for general grounded questions |
| 13–14 | 47–51 | Banking tests (agents never seen in training), confusion matrix, "ding-dong" conversations that jump between agents |
| 15 | 58–59 | Summary, and five brand-new conversations routed live |
| 16 | 60–72 | A yes / no **context-change detector**, the "confident stay" gate, and the realistic tests: E10 (chat history), E11 (first message), E12 (grounded questions) |
| 17 | — | **What we tried and dropped**, with the numbers |

**The final design of this notebook** (the best frozen system, runs 10–17):

```text
ModernBERT-large → pooled spans → MLP-256 → p(topic changed)
   p < τ  → keep the current agent
   p ≥ τ  → Nemotron-1B (cross + dual passes) → matching features → MLP-256 → one agent
```

It beat the 2B Strands decider on routing in conversations at unseen companies (74 / 83 vs 62 with full history), but general choice questions plateaued at 114 / 170 against the decider's 160. **The frozen encoder was the limit**, and Phase 17 shows the evidence. `03_lora_signpost.ipynb` (branch `signpost`) fine-tunes the encoder with LoRA and closes that gap.

**Runs on:** Kaggle with a GPU (T4 or better). Training data comes from this branch (`data/modernbert_router/`). The numbers recorded in the experiment ledger (`experiments/modernbert_router_ledger.md`, runs 5–17) were produced at each phase's own commit with that phase's data. This notebook runs the final state on the final data, so its numbers will be close to, but not identical with, run 17's.

**Settings** (environment variables, all optional): `ROUTER_ENCODER` (default `nemotron-1b`), `DETECTOR_ENCODER` (default `modernbert`), `REALISTIC_TRAIN` (1), `CHOICE_TRAIN` (1), `ROUTER_SMOKE=1` for a fast code check only.

In [ ]:
# Setup — packages, and the ModernBERT pieces this notebook takes from 01_modernbert_by_hand.ipynb

!pip install -q "transformers==5.17.0"

import json
import sys
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoTokenizer

print("Python :", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 01 rebuilt this checkpoint's forward pass by hand and matched the library to ~3e-5. From here on the library runs it
# (Cell 34b), which is faster and frees us to load a second encoder (Nemotron-1B) the same way.
CHECKPOINT = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
config = AutoConfig.from_pretrained(CHECKPOINT)
SEP_ID = tokenizer.sep_token_id
H = config.hidden_size                                              # 1,024 for ModernBERT-large


def mean_pool(hidden, attention_mask):
    """Mask-aware mean pooling (01, Cell 20): [B, S, H] -> [B, H], padding excluded."""
    m = attention_mask[..., None].to(hidden.dtype)                  # [B, S, 1]
    return (hidden * m).sum(dim=1) / m.sum(dim=1)                   # [B, H]


print(f"{CHECKPOINT}: H = {H}, {config.num_hidden_layers} layers | [SEP] id {SEP_ID}")

---
# Phase 11 — Compatibility model

## Cell 34 — Define the routing problem

**What.** Load the routing data and state the problem formally:

```text
input:   chat history  +  one candidate agent description
output:  one compatibility score

f(conversation, agent) → score
```

**Why.** The router must choose which agent should handle the conversation **now**. Scoring each candidate separately with the same function `f` means the candidates are data, not architecture. That's the property Cell 33 showed the fixed classifier lacks.

**Generic training, banking testing.** The router is trained **only on generic conversations from other domains**, never on the five banking agents. If it then routes banking conversations correctly, it has learned *compatibility between a conversation and an agent description*, not five memorized agents.

| File (under `data/modernbert_router/`) | Role | Contents |
|---|---|---|
| `generic/domains.json` + `generic/conversations.json` | **training and validation** | template conversations from **non-banking domains** (IT helpdesk, HR, airline, clinic, hotel, gym, library, streaming, shipping, school, vet, cloud hosting, wedding planning, …), each domain with its own 5 agents |
| `realistic_train/` | **training** | free-form realistic conversations (50 businesses), added in Phase 6 (`REALISTIC_TRAIN=1`) |
| `choice_train/items.json` | **training (choice head only)** | general grounded questions: yes / no, choice and score (`CHOICE_TRAIN=1`) |
| `banking/benchmark.json` | **test** | 50 held-out banking conversations (2 per seed scenario, new wording) |
| `banking/extended.json` | **test** | 175 more banking conversations (the 25 seed scenarios + 6 paraphrases each) |
| `banking/dingdong.json` | **test** | 5 banking ding-dong conversations, with the expected agent after every user turn (Phase 14) |

There is **no fixed validation split**. Phase 12 validates by leaving whole domains out: the head is always tested on domains, and therefore agents, it hasn't seen.

Every domain has one deliberately **close pair** of agents (for example, Password & Access vs Network & VPN) and one **general** agent, mirroring Payments vs Card Support and General Customer Support in banking. The generic data covers the same patterns the spec requires:
- obvious requests and close-pair clarification
- handoff + "OK" and handoff chains
- follow-ups and "okay" messages
- **topic changes where the latest request wins**
- **ding-dong walks labelled after every user turn**
- distractor keywords, typos and long conversations

The generators are in `data/modernbert_router/`. All conversations are in LangGraph message format, with 1–15 user/assistant messages and no tools.

**Where it runs.**
- **Data:** the cell uses the local `data/modernbert_router` folder if it exists. Otherwise it `git clone`s the repo at the branch named by `DATA_BRANCH`, which is what happens on Kaggle or Colab. Each experiment lives on its own branch, so its data comes from that branch. An existing clone is refreshed, so a rerun never uses stale data.
- **Device:** the **GPU** if one is available. The router needs thousands of encoder passes.
- **Current agent:** every conversation records `current_agent`, the agent that spoke the last assistant message (none for a fresh conversation). It's the agent holding the conversation when the router is called. Often it is **not** the right answer, for example after a handoff or a topic change, so the router must learn when to stay and when to switch.
- **Progress logging:** long-running steps print timestamped progress lines through `log()`, and also append them to `router_run.log`, so a long run can be followed while it's going.
- **Smoke test:** setting the environment variable `ROUTER_SUBSAMPLE` (e.g. 6) uses only that many conversations per domain, for a quick end-to-end check on a CPU.

**Expect.**
- The device in use.
- A table of the 27 domains, with their agents and conversation counts.
- Confirmation that no generic conversation mentions a banking agent.
- Banking test counts and one generic example conversation.

In [ ]:
# Cell 34 — The routing problem: f(conversation, agent) -> score; load generic training data + banking test data

import gc
import os
import subprocess
from collections import Counter
from pathlib import Path

# ---- Device: the router's thousands of encoder passes run on the GPU if one is present ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Router computations run on: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))

# ---- Data: use the local repo folder, or clone the repo (e.g. on Kaggle/Colab) ----
# On Kaggle/Colab the data is cloned from this notebook's branch.
DATA_BRANCH = "frozen-router"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:                                      # refresh an existing clone, so a rerun never trains on stale data
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"
print("data folder:", DATA_DIR.resolve(), "| branch:", DATA_BRANCH)

# Progress logging for the long-running steps: printed live and appended to router_run.log
RUN_LOG = Path("router_run.log")


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def mem():
    """Host RAM used by this process and GPU memory allocated, for the progress log."""
    import psutil
    gpu = f" | GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB" if torch.cuda.is_available() else ""
    return f"RAM {psutil.Process().memory_info().rss / 1e9:.1f} GB{gpu}"


log(f"router run started on {DEVICE} | {mem()}")

# 0 = use every generic conversation; a small number (e.g. 6 per domain) gives a quick CPU smoke test
ROUTER_SUBSAMPLE = int(os.environ.get("ROUTER_SUBSAMPLE", "0"))
# ROUTER_SMOKE=1: a fast code check only (small test sets, one head type); never set it for a real run
ROUTER_SMOKE = os.environ.get("ROUTER_SMOKE") == "1"


def load(path):
    with open(path) as f:
        return json.load(f)


generic_domains = load(DATA_DIR / "generic" / "domains.json")
generic_conversations = load(DATA_DIR / "generic" / "conversations.json")
# Phase 6: free-form realistic conversations (40 businesses, 744 user turns) trained alongside the templates.
# Their businesses and agent names appear in no test set; REALISTIC_TRAIN=0 trains on the templates alone.
REALISTIC_TRAIN = os.environ.get("REALISTIC_TRAIN", "1") == "1"
if REALISTIC_TRAIN:
    realistic_domains = load(DATA_DIR / "realistic_train" / "domains.json")
    realistic_train = load(DATA_DIR / "realistic_train" / "conversations.json")
    assert not set(realistic_domains) & set(generic_domains)
    generic_domains = {**generic_domains, **realistic_domains}
    generic_conversations = generic_conversations + realistic_train
    print(f"realistic training data: {len(realistic_domains)} domains, {len(realistic_train)} conversations added")
if ROUTER_SUBSAMPLE:                      # evenly spread over each domain, so every kind of conversation is represented
    by_domain = {}
    for ex in generic_conversations:
        by_domain.setdefault(ex["domain"], []).append(ex)
    generic_conversations = [convs[i * len(convs) // ROUTER_SUBSAMPLE] for convs in by_domain.values() for i in range(ROUTER_SUBSAMPLE)]
    print(f"SMOKE TEST: {ROUTER_SUBSAMPLE} conversations per domain")
# Run 17: the same input format for every question, (text, question, options), but TWO SEPARATE HEADS that never share data:
# a route head (routing data and tests only) and a choice head (general questions and E12 only). CHOICE_TRAIN=0: route head only.
ROUTE_Q = "Which agent should handle the user's latest message?"
CHANGE_Q = "Does the latest message move the conversation to a different topic?"
CHOICE_TRAIN = os.environ.get("CHOICE_TRAIN", "1") == "1"
choice_train = load(DATA_DIR / "choice_train" / "items.json")["items"] if CHOICE_TRAIN else []
if ROUTER_SUBSAMPLE or ROUTER_SMOKE:
    choice_train = choice_train[::25]
print(f"choice training items: {len(choice_train)} (" + ", ".join(f"{t} {n}" for t, n in Counter(x["type"] for x in choice_train).items()) + ")")
banking_benchmark = load(DATA_DIR / "banking" / "benchmark.json")
banking_extended = load(DATA_DIR / "banking" / "extended.json")
router_dingdong = load(DATA_DIR / "banking" / "dingdong.json")
if ROUTER_SMOKE:
    banking_benchmark, banking_extended, router_dingdong = banking_benchmark[:10], banking_extended[:5], router_dingdong[:2]
    print("SMOKE CHECK: 10 + 5 banking conversations, 2 ding-dong sequences")
banking_all = banking_benchmark + banking_extended
benchmark = banking_benchmark

missing = [ex["id"] for ex in generic_conversations + banking_all if "current_agent" not in ex]
assert not missing, f"data without current_agent (pull the latest data / rerun build_generic_router_data.py): {missing[:3]}"
missing = [ex["id"] for ex in generic_conversations if "context_change" not in ex]
assert not missing, f"generic data without context_change (pull the latest data): {missing[:3]}"

for ex in generic_conversations + banking_all:
    roles = [m["role"] for m in ex["messages"]]
    assert 1 <= len(roles) <= 15 and roles[0] == "user" and roles[-1] == "user", ex["id"]
    assert all(r1 != r2 for r1, r2 in zip(roles, roles[1:])), "roles alternate (chronological turns)"

banking_names = ["payments agent", "card support", "investment agent", "insurance agent", "general customer support"]
leaks = [ex["id"] for ex in generic_conversations
         if any(w in " ".join(m["content"].lower() for m in ex["messages"]) for w in banking_names)]
assert not leaks, leaks
print("No generic conversation mentions a banking agent.\n")

print(f"{'domain':20s} | {'conversations':>13} | agents (close pair marked *)")
for dom, info in generic_domains.items():     # (not "d": that name holds head_dim for the encoder)
    n = sum(ex["domain"] == dom for ex in generic_conversations)
    names = ", ".join(a + ("*" if a in info["close_pair"] else "") for a in info["agents"])
    print(f"{dom:20s} | {n:>13} | {names}")

print(f"\ngeneric: {len(generic_domains)} domains, {len(generic_conversations)} conversations")
print(f"banking TEST: benchmark {len(banking_benchmark)} + extended {len(banking_extended)} = {len(banking_all)} conversations; "
      f"ding-dong {len(router_dingdong)} sequences / {sum(len(s['user_turns']) for s in router_dingdong)} user turns")

example = next((ex for ex in generic_conversations if "ok_after_handoff" in ex["tags"] and len(ex["messages"]) == 5),
               generic_conversations[0])
print(f"\nGeneric example {example['id']} (tags {example['tags']}):")
for m in example["messages"]:
    print(f"  {m['role']:9s}: {m['content']}")
print(f"  -> current agent (spoke last): {example['current_agent']} | target agent: {example['target_agent']}")
stay = sum(ex["current_agent"] == ex["target_agent"] for ex in generic_conversations)
fresh = sum(ex["current_agent"] is None for ex in generic_conversations)
print(f"\ngeneric current agent: none {fresh} | target = current (stay) {stay} | target != current (switch) {len(generic_conversations) - stay - fresh}")

## Cell 34b — Two encoders: one for the router, one for the Y/N detectors

**What.** The router (Cells 44–59) and the Y/N context-change detector (Cells 60–63) each read text through **their own encoder**, chosen independently:

| `ROUTER_ENCODER` \ `DETECTOR_ENCODER` | `modernbert` | `nemotron-1b` |
|---|---|---|
| `modernbert` | as in run 8 (3a) | new |
| `nemotron-1b` | **the hybrid**: best router (run 9) + best detectors (run 8) | as in run 9 (3b) |

Set them at the top of this cell (or with the environment variables of the same names).

- **`modernbert`** is ModernBERT-large through the library `AutoModel`, in float32 (01 matched our own forward pass to it), separator `" [SEP] "`, no prefix.
- **`nemotron-1b`** uses the library `AutoModel` (`nvidia/llama-nemotron-embed-1b-v2`, trust_remote_code), with `query:` / `passage:` prefixes, newline separators, fp16 on the GPU.
- An encoder used by both sides is loaded **once**.

**Spans by characters.** `build_text` records the character range of every part in the chosen encoder's format, and `char_mask` selects the tokens that overlap a range through the tokenizer's offset mapping. That works for any tokenizer.

**Caches** are kept per encoder, so the four combinations never mix features.

**Expect.** Both encoders, whether they share one instance, an example router input text, and the character ranges of its parts.

In [ ]:
# Cell 34b — Two encoders, chosen independently: one for the ROUTER, one for the Y/N DETECTORS (ModernBERT ours / Nemotron library)

ENCODERS = {
    # ModernBERT-large through the library; 01 rebuilt this forward pass by hand and matched it to ~3e-5.
    # float32, as our own forward pass ran in the experiments. " [SEP] " is ModernBERT's separator token
    "modernbert":   {"forward": "library", "checkpoint": CHECKPOINT, "trust_remote_code": False, "dtype": torch.float32,
                     "sep": " [SEP] ", "query": "", "passage": ""},
    # library forward passes (AutoModel). Nemotron is bidirectional Llama-3.2-1B, trained for retrieval with these prefixes
    "nemotron-1b":  {"forward": "library", "checkpoint": "nvidia/llama-nemotron-embed-1b-v2", "trust_remote_code": True,
                     "sep": "\n", "query": "query: ", "passage": "passage: "},
    # a small Llama-family model, only to run the library code path on a CPU (it is causal, so its numbers mean nothing)
    "smollm2-135m": {"forward": "library", "checkpoint": "HuggingFaceTB/SmolLM2-135M", "trust_remote_code": False,
                     "sep": "\n", "query": "query: ", "passage": "passage: "},
}
# The four combinations: router in {modernbert, nemotron-1b} x detector in {modernbert, nemotron-1b}
ROUTER_ENCODER = os.environ.get("ROUTER_ENCODER", "nemotron-1b")
DETECTOR_ENCODER = os.environ.get("DETECTOR_ENCODER", "modernbert")
MIX_LAYERS = 4                                   # "mix" = mean of the last 4 layers' outputs (Phase 3 layer-mix option)
LAYERS = ["last"]                                # "mix" dropped (it lost everywhere in run 8 / 3a)


class TextEncoder:
    """One encoder: its tokenizer, text format (separator, prefixes), forward pass and cache tag."""

    def __init__(self, key):
        spec = ENCODERS[key]
        self.key, self.sep, self.query, self.passage, self.forward = key, spec["sep"], spec["query"], spec["passage"], spec["forward"]
        from transformers import AutoModel
        self.name = spec["checkpoint"]
        self.tokenizer = AutoTokenizer.from_pretrained(self.name, trust_remote_code=spec["trust_remote_code"])
        self.tokenizer.padding_side = "right"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        dtype = spec.get("dtype") or (torch.float16 if DEVICE.type == "cuda" else torch.float32)
        self.model = AutoModel.from_pretrained(self.name, trust_remote_code=spec["trust_remote_code"], dtype=dtype).to(DEVICE).eval()
        self._captured = []
        for layer in self.model.layers[-MIX_LAYERS:]:                     # catch the last layers' outputs for the layer mix
            layer.register_forward_hook(lambda mod, inp, out: self._captured.append(out[0] if isinstance(out, tuple) else out))
        self.tag = self.name.split("/")[-1] + "_p3"                          # caches per encoder (and per feature layout)

    def run(self, input_ids, attention_mask):
        """-> (final output [B, T, H], mean of the last MIX_LAYERS layer outputs [B, T, H])."""
        self._captured.clear()
        out = self.model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return out.float(), torch.stack(self._captured[-MIX_LAYERS:]).float().mean(0)


_ENCODER_INSTANCES = {}


def get_encoder(key):
    """Load each encoder once, even when the router and the detectors use the same one."""
    if key not in _ENCODER_INSTANCES:
        _ENCODER_INSTANCES[key] = TextEncoder(key)
    return _ENCODER_INSTANCES[key]


ROUTER_ENC, DETECTOR_ENC = get_encoder(ROUTER_ENCODER), get_encoder(DETECTOR_ENCODER)
# the router side keeps its earlier names (Cells 35, 44a, 45c read them)
ENC = ENCODERS[ROUTER_ENCODER]
enc_tokenizer, enc_name, MODEL_TAG = ROUTER_ENC.tokenizer, ROUTER_ENC.name, ROUTER_ENC.tag


def build_text(messages, current=None, candidate=None, prefix="", enc=None, question=None):
    """The input text and the CHARACTER range of every part, in the format of encoder `enc` (default: the router's):
    prefix + msg_1 SEP msg_2 ... SEP "current agent: X" [SEP "question: Q"] [SEP "candidate: <option>"]
    Without a question (the ModernBERT detector's input) the candidate is written "candidate agent: ...", as before run 16.
    -> (text, {"messages": [(start, end), ...], "current": ..., "question": ... or None, "candidate": ... or None})"""
    sep = (enc or ROUTER_ENC).sep
    text, spans = prefix, {"messages": [], "current": None, "question": None, "candidate": None}
    for k, m in enumerate(messages):
        text += sep if k else ""
        seg = f"{m['role']}: {m['content']}"
        spans["messages"].append((len(text), len(text) + len(seg))); text += seg
    cand_label = "candidate agent" if question is None else "candidate"
    for key, seg in [("current", f"current agent: {current or 'none'}"),
                     ("question", None if question is None else f"question: {question}"),
                     ("candidate", None if candidate is None else f"{cand_label}: {candidate}")]:
        if seg is not None:
            text += sep
            spans[key] = (len(text), len(text) + len(seg)); text += seg
    return text, spans


def encode_batch(texts, enc=None):
    """Tokenize with character offsets and run encoder `enc` (default: the router's)
    -> (last [B,T,H], mix [B,T,H], attention mask [B,T], offsets [B,T,2], ids)."""
    enc = enc or ROUTER_ENC
    batch = enc.tokenizer(texts, padding=True, return_tensors="pt", return_offsets_mapping=True)
    offsets = batch.pop("offset_mapping")
    with torch.no_grad():
        last, mix = enc.run(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
    assert torch.isfinite(last).all() and torch.isfinite(mix).all(), \
        f"{enc.name} produced inf/NaN (fp16 overflow?): load it in float32 in this cell instead"
    return last, mix, batch["attention_mask"], offsets, batch["input_ids"]


def char_mask(offsets, attention, rng):
    """Tokens of one text that overlap the character range rng = (start, end); rng None = every real (non-special) token."""
    s, e = offsets[:, 0], offsets[:, 1]
    real = attention.bool() & (e > s)
    return real if rng is None else real & (s < rng[1]) & (e > rng[0])


def span_means(texts, ranges, cache, cache_file, title, mix=True, enc=None):
    """Mean of the encoder's token vectors over each character range, cached by text.
    texts: N strings; ranges: N lists of S ranges -> [N, L, S, H] with L = 2 (last, mix) or 1 (last)."""
    enc = enc or ROUTER_ENC
    todo = [t for t in dict.fromkeys(texts) if t not in cache]
    if todo:
        rng_of = {t: r for t, r in zip(texts, ranges)}
        t0 = time.perf_counter()
        verbose = len(todo) >= 500                         # log only the big feature runs, not live routing
        if verbose:
            log(f"{title}: {len(todo)} new encoder passes ({enc.name})")
        order = sorted(todo, key=len)
        for start in range(0, len(order), BATCH):
            if verbose and start and start % (BATCH * 100) == 0:
                el = time.perf_counter() - t0
                log(f"  {title} {start}/{len(todo)} | {start / el:.0f} texts/s | ~{(len(todo) - start) / (start / el):.0f}s left")
            chunk = order[start:start + BATCH]
            last, mixh, attn, offs, _ = encode_batch(chunk, enc)
            for row, t in enumerate(chunk):
                masks = [char_mask(offs[row], attn[row], r) for r in rng_of[t]]
                masks = [m if m.any() else char_mask(offs[row], attn[row], None) for m in masks]      # never an empty span
                per_layer = [torch.stack([h[row][m.to(DEVICE)].mean(0) for m in masks]) for h in ([last, mixh] if mix else [last])]
                cache[t] = torch.stack(per_layer).half().cpu()
        timing["encoder_passes"] += len(todo); timing["encoder_seconds"] += time.perf_counter() - t0
        if verbose:                                        # persist only big runs; live routing stays in memory
            torch.save(cache, cache_file)
            log(f"{title} done: {len(todo)} passes in {time.perf_counter() - t0:.0f}s (cached)")
    return torch.stack([cache[t] for t in texts]).float()


timing = {"encoder_passes": 0, "encoder_seconds": 0.0}
BATCH = 32 if DEVICE.type == "cuda" else 8
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

demo_text, demo_spans = build_text(example["messages"], example["current_agent"], "Some Agent. Handles something.", ENC["query"],
                                   question=ROUTE_Q)
print(f"ROUTER   encoder: {ROUTER_ENCODER} ({ROUTER_ENC.name}, {ROUTER_ENC.forward} forward pass)")
print(f"DETECTOR encoder: {DETECTOR_ENCODER} ({DETECTOR_ENC.name}, {DETECTOR_ENC.forward} forward pass)"
      f"{'  [same instance as the router]' if DETECTOR_ENC is ROUTER_ENC else ''}")
print(f"layer options: {LAYERS} | {mem()}")
print(f"\nexample router input text:\n{demo_text}\n")
print({k: v for k, v in demo_spans.items()})

## Cell 35 — Construct the cross-encoder input

**What.** Serialize a conversation plus one candidate agent into a **single** text, which becomes **one** ModernBERT input:

```text
[CLS] user: … [SEP] assistant: … [SEP] user: … [SEP] … candidate agent: <name>. <description> [SEP]
```

**Why.** In a **cross-encoder**, the conversation and the candidate are encoded **together**. Because ModernBERT's attention is fully bidirectional (Phase 5), every conversation token can attend to every description token and vice versa, *inside every layer*. The model can compare, for example, "the card itself says blocked" directly against "card blocking and unblocking" (or, in the generic data, "it rejects my login" against "sign-in errors").

**Rules.**
- Keep **both** user and assistant messages, because handoffs ("Please continue with the Payments Agent") are assistant messages.
- Keep chronological order, prefix each message with its role, and allow up to 15 messages.
- Tools are ignored: there are none in this dataset.
- `[SEP]` separates the messages and the candidate. The tokenizer adds the outer `[CLS]` and final `[SEP]`.

**Shapes.** One pair → `input_ids` `[1, S]`, where `S` grows with conversation length. Even 15-message conversations stay far below ModernBERT's 8,192-token limit.

**Expect.** The token sequence starts with `[CLS]`. It contains exactly one `[SEP]` per message plus the final `[SEP]`, and the candidate description is at the end.

**Current agent.** Between the history and the candidate sits one more segment, `current agent: <name>` (or `none`). It names the agent that spoke the last assistant message. In the joint pass, the candidate's tokens can attend to it, so "the candidate **is** the current agent" is visible inside the encoder. A short reply like "ok" or "Sunday" usually belongs to whoever is holding the conversation, unless the last assistant message handed it to someone else.

In [ ]:
# Cell 35 — Cross-encoder input: conversation turns + candidate agent, as one sequence

MAX_MESSAGES = 15


def serialize(messages, candidate_text, current=None, question=ROUTE_Q):
    """history SEP current agent SEP question SEP candidate: one segment per message, who holds the chat, what is asked,
    then the option. The separator and any prefix come from the router encoder (Cell 34b)."""
    assert 1 <= len(messages) <= MAX_MESSAGES
    return build_text(messages, current, candidate_text, ENC["query"], question=question)[0]


sample_candidate = f"{example['target_agent']}. {generic_domains[example['domain']]['agents'][example['target_agent']]}"
pair_text = serialize(example["messages"], sample_candidate, example["current_agent"])
pair_ids = enc_tokenizer(pair_text)["input_ids"]
pair_tokens = enc_tokenizer.convert_ids_to_tokens(pair_ids)

print(pair_text[:400], "...")
print(f"\ntokens: {len(pair_ids)} | first: {pair_tokens[0]} | last: {pair_tokens[-1]}")
print("token sequence (first 40):", pair_tokens[:40])

if ROUTER_ENCODER == "modernbert":                              # ModernBERT: the separators are real [SEP] tokens
    n_sep = pair_ids.count(SEP_ID)
    assert pair_tokens[0] == "[CLS]" and pair_tokens[-1] == "[SEP]"
    assert n_sep == len(example["messages"]) + 3, "one [SEP] per message + after the current agent + after the question + final"
    assert len(pair_ids) < config.max_position_embeddings
    print(f"[SEP] tokens: {n_sep} = {len(example['messages'])} messages + current agent + question + final")

## Cell 36 — Compatibility head

**What.** Put a **one-output** linear head on top of ModernBERT:

```text
(conversation ⊕ candidate)  →  ModernBERT  →  mean pool  →  Linear(1024 → 1)  →  score
```

**Why.** This is a **head over ModernBERT**, not a separate model. ModernBERT does all the language work, reading the conversation and the description together. The head only learns which directions of the pooled representation indicate "this agent fits this conversation". Because it outputs a single number for any (conversation, candidate) pair, the number of candidates is unlimited and determined at run time.

**Math.**

```text
v = mean_pool(ModernBERT(serialize(conversation, agent)))      v ∈ ℝ^1024
score = w · v + b                                              w ∈ ℝ^1024, b ∈ ℝ  (1,025 parameters)
```

**Shapes.** A batch of `N` pairs → `input_ids` `[N, S]` → encoder output `[N, S, 1024]` → pooled `[N, 1024]` → scores `[N]`.

`encode_pairs` sorts texts by length before batching, so each batch pads as little as possible, then restores the original order. The scores are identical either way.

**Expect.** The head has 1,025 parameters. An (untrained) score is printed for the example pair.

In [ ]:
# Cell 36 — The compatibility head: Linear(H -> 1) over mean-pooled ModernBERT output

torch.manual_seed(42)
compat_head = nn.Linear(H, 1)


def encode_pairs(texts, batch_size=32 if DEVICE.type == "cuda" else 8):
    """Run ModernBERT (the library forward pass, Cell 34b) on each text and mean-pool: list of N strings -> [N, H]."""
    mb = get_encoder("modernbert")
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))           # similar lengths together
    pooled = [None] * len(texts)
    with torch.no_grad():
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            batch = mb.tokenizer([texts[i] for i in idx], padding=True, return_tensors="pt").to(DEVICE)
            out = mb.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state   # [b, S, H]
            for i, v in zip(idx, mean_pool(out, batch["attention_mask"]).cpu()):
                pooled[i] = v
    return torch.stack(pooled)                                                          # [N, H]


pair_vec = encode_pairs([pair_text])                                 # [1, H]
with torch.no_grad():
    pair_score = compat_head(pair_vec).squeeze(-1)                    # [1]

print("pooled pair representation:", tuple(pair_vec.shape))
print("head weight:", tuple(compat_head.weight.shape), "| bias:", tuple(compat_head.bias.shape),
      f"| parameters: {sum(p.numel() for p in compat_head.parameters())}")
print(f"untrained score for (example conversation, Payments Agent): {pair_score.item():.4f}")

## Cell 37 — Five candidate agents

**What.** Define the five agents. Each description is part of the model input.

**Why.** The router can only be as good as the distinctions the descriptions make. Two agents are **deliberately close**:

- **Payments Agent:** the *transaction*. Transfers, failed or duplicate payments, refunds, settlement.
- **Card Support Agent:** the *card*. Blocking, replacement, limits, PIN, security of card details.

A message such as "my card payment failed" contains keywords of both, so keyword matching can't decide. The router has to read *what the customer actually needs*, often from earlier turns.

**Shapes.** Five `(name, description)` pairs. Each candidate text is `"<name>. <description>"`, which adds about 40–70 tokens to every cross-encoder input.

**Expect.** A table of the five agents and the token length of each candidate text.

In [ ]:
# Cell 37 — The five candidate agents (descriptions are part of the model input)

AGENTS = {
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
AGENT_NAMES = list(AGENTS)
CLOSE_PAIR = {"Payments Agent", "Card Support Agent"}


def agent_text(name):
    return f"{name}. {AGENTS[name]}"


assert len(AGENT_NAMES) == 5
for name in AGENT_NAMES:
    n_tok = len(tokenizer(agent_text(name), add_special_tokens=False)["input_ids"])
    close = "  <- closely related pair" if name in CLOSE_PAIR else ""
    print(f"{name:32s} {n_tok:3d} tokens{close}")
    print(f"    {AGENTS[name]}")

## Cell 38 — Score all five agents

**What.** For one conversation, run ModernBERT **five times**, once per candidate agent:

```text
history + Payments Agent        → ModernBERT → score A
history + Card Support Agent    → ModernBERT → score B
history + Investment Agent      → ModernBERT → score C
history + Insurance Agent       → ModernBERT → score D
history + General Support Agent → ModernBERT → score E
```

**Why five separate passes?** This is the defining property of a **cross-encoder**. The conversation is re-encoded *together with* each candidate, so every layer can compare the two. The conversation's representation is therefore different for each candidate, and nothing can be shared between the five passes. That's the cost of letting conversation and description attend to each other.

**Shapes.** Each pass is `[1, S_k]` → `[1, S_k, 768]` → `[768]` → one score, where `S_k` differs slightly per agent. The five scores together form `[5]`.

**Expect.** Five scores, plus the time each encoder pass took on this CPU. The head isn't trained yet, so the scores aren't meaningful.

In [ ]:
# Cell 38 — Five deliberate encoder passes: one per (conversation, candidate agent)

routing_example = next((ex for ex in banking_benchmark if ex["id"] == "011-e0"), banking_benchmark[0])   # banking test: handoff + "OK"
print("Conversation:")
for m in routing_example["messages"]:
    print(f"  {m['role']:9s}: {m['content']}")

raw_scores, pass_times = [], []
for name in AGENT_NAMES:
    text = serialize(routing_example["messages"], agent_text(name), routing_example["current_agent"])
    t0 = time.perf_counter()
    vec = encode_pairs([text])                                       # ONE full ModernBERT pass
    pass_times.append(time.perf_counter() - t0)
    with torch.no_grad():
        raw_scores.append(compat_head(vec).squeeze())
    print(f"  pass for {name:32s}: {len(tokenizer(text)['input_ids']):3d} tokens, {pass_times[-1] * 1000:6.0f} ms")

untrained_scores = torch.stack(raw_scores)                           # [5]
print("\nscores:", tuple(untrained_scores.shape), untrained_scores)
print(f"encoder passes: {len(AGENT_NAMES)} | total time {sum(pass_times):.2f}s")

## Cell 39 — Softmax

**What.** Turn the five raw scores into routing probabilities and pick the most likely agent:

```python
probabilities = torch.softmax(scores, dim=0)
selected = argmax(probabilities)
```

**Why.** Raw scores are on an arbitrary scale. Softmax makes them comparable *within this decision*: the five probabilities are positive and sum to 1. The argmax is the routed agent. The probability margin gives a confidence signal that later cells use (for example, when deciding whether to switch agents).

**Math.** `p_k = exp(s_k) / Σ_j exp(s_j)` over the five candidates. Softmax is shift-invariant: adding a constant to every score changes nothing. Only the **differences** between candidates matter.

**Shapes.** `scores` `[5]` → `probabilities` `[5]` → `selected` (a scalar index).

**Expect.** Five probabilities summing to 1 and an argmax index. The head is still untrained, so the selection is arbitrary.

In [ ]:
# Cell 39 — Scores -> routing probabilities -> selected agent

probabilities = torch.softmax(untrained_scores, dim=0)              # [5]
selected = int(torch.argmax(probabilities))

print("probabilities:", probabilities)
print(f"sum = {probabilities.sum():.6f} | selected index {selected} -> {AGENT_NAMES[selected]}")
assert torch.isclose(probabilities.sum(), torch.tensor(1.0))
assert torch.allclose(torch.softmax(untrained_scores + 7.0, dim=0), probabilities), "shift-invariant"

## Cell 40 — Inspect the routing decision

**What.** Show the complete decision as a table:

```text
Agent | Raw score | Probability | Selected?
```

**Why.** We'll use this view throughout the evaluation. It shows not only which agent won but also by how much, and which agent was runner-up. For the Payments / Card Support pair, the runner-up is often the more informative part.

**Expect.** One row per agent. The correct answer for this banking test conversation is the **Payments Agent**: the user said "OK" right after being handed off to Payments. The untrained head has no reason to pick it, and training (Phase 12) is what changes that.

In [ ]:
# Cell 40 — The routing decision as a table

def decision_table(names, scores, correct=None):
    probs = torch.softmax(scores, dim=0)
    best = int(torch.argmax(scores))
    print(f"{'Agent':34s} | {'Raw score':>9} | {'Probability':>11} | Selected?")
    print("-" * 72)
    for k, name in enumerate(names):
        mark = "  <== selected" if k == best else ""
        flag = "  (correct)" if correct == name else ""
        print(f"{name:34s} | {scores[k].item():9.4f} | {probs[k].item():11.3f} |{mark}{flag}")
    return names[best]


chosen = decision_table(AGENT_NAMES, untrained_scores, correct=routing_example["target_agent"])
print(f"\nuntrained head selects: {chosen} | correct: {routing_example['target_agent']}")

## Cell 41 — Two heads, two training sets, never mixed

**Run 17.** All questions use the same input format and the same frozen encoder:

```text
query: <conversation or text> | current agent: X | question: <what is asked> | candidate: <one option>
```

But there are **two separate heads**. They share no training data, no folds, no statistics and no tests:

| | Route head | Choice head |
|---|---|---|
| Training data | 6,850 routing conversations ("Which agent should handle the user's latest message?") | 1,524 general grounded questions (`choice_train/items.json`: yes / no, choice, score) |
| Leave-domains-out folds | over the routing domains | over the 11 question families |
| Feature standardization | routing pairs only | choice pairs only |
| Tests | E10 (+ gate), E11, banking, ding-dong, Cell 59 | E12 |

The context-change labels are used by neither head. The gate keeps the separate ModernBERT detector (Cells 60–63).

**Why.** Run 16 trained one head on everything. This run checks whether each job does better with a head of its own: the route head is compared with run 16 on the routing tests, and the choice head with run 16 on E12.

**Expect.** Examples, pairs and folds per head, and one serialized example per task.


In [ ]:
# Cell 41 — Two training sets that are never mixed: ROUTING (route head) and GENERAL CHOICE (choice head)

import random

K_MAX = 5                                                   # options per question: 2 (yes / no) up to 5 (agents)
CHOICE_TASKS = {"noul", "choice", "score"}


def option_text(name, description):
    return f"{name}. {description}" if description else name


def routing_examples():
    """Routing: which agent should handle the latest message? -> {id, domain, task, messages, current, question, options, target}."""
    return [{"id": ex["id"], "domain": ex["domain"], "task": "routing", "messages": ex["messages"], "current": ex["current_agent"],
             "question": ROUTE_Q, "options": generic_domains[ex["domain"]]["agents"], "target": ex["target_agent"]}
            for ex in generic_conversations]


def general_choice_examples():
    """General grounded questions about a short text (yes / no, choice, score)."""
    return [{"id": it["id"], "domain": it["domain"], "task": it["type"], "messages": [{"role": "user", "content": it["state"]}],
             "current": None, "question": it["question"], "options": {o: "" for o in it["options"]}, "target": it["answer"]}
            for it in choice_train]


def make_pairs(examples):
    """One (example, option) pair per option; 'ex' and 'slot' place it on the [examples, K_MAX] grid."""
    pairs = []
    for ei, ex in enumerate(examples):
        assert 2 <= len(ex["options"]) <= K_MAX and ex["target"] in ex["options"], ex["id"]
        for slot, (name, description) in enumerate(ex["options"].items()):
            cand = option_text(name, description)
            pairs.append({"ex": ei, "slot": slot, "conv_id": ex["id"], "domain": ex["domain"], "agent": name, "target": ex["target"],
                          "label": float(name == ex["target"]), "messages": ex["messages"], "current": ex["current"],
                          "question": ex["question"], "candidate": cand,
                          "text": serialize(ex["messages"], cand, ex["current"], ex["question"])})
    return pairs


route_examples, choice_examples = routing_examples(), general_choice_examples()
# One encoding pass over every text (the encoder is frozen and learns nothing); training, folds, statistics and tests
# are then kept apart per group.
train_examples = route_examples + choice_examples
ex_group = ["route"] * len(route_examples) + ["choice"] * len(choice_examples)
generic_pairs = make_pairs(train_examples)
conv_domain = [ex["domain"] for ex in train_examples]
ex_task = [ex["task"] for ex in train_examples]
GROUPS = ["route", "choice"] if choice_examples else ["route"]

N_FOLDS = 5
FOLDS = {}
for g in GROUPS:                                            # leave-domains-out folds WITHIN each group
    doms = sorted({d for d, gg in zip(conv_domain, ex_group) if gg == g})
    random.Random(42).shuffle(doms)
    FOLDS[g] = [doms[i::N_FOLDS] for i in range(N_FOLDS)]

for g in GROUPS:
    n_ex = ex_group.count(g)
    n_pairs = sum(ex_group[p_["ex"]] == g for p_ in generic_pairs)
    print(f"{g:6s} head: {n_ex} examples -> {n_pairs} pairs | tasks {dict(Counter(t for t, gg in zip(ex_task, ex_group) if gg == g))}")
    for f, doms in enumerate(FOLDS[g]):
        print(f"   fold {f}: {sum(dom in doms for dom, gg in zip(conv_domain, ex_group) if gg == g):5d} examples | {len(doms)} domains")

for task in ["routing", "noul", "choice", "score"]:
    ex = next((e for e in train_examples if e["task"] == task), None)
    if ex:
        print(f"\n[{task}] {ex['id']}: question {ex['question']!r} | options {list(ex['options'])} | target {ex['target']}")
        print("  " + serialize(ex["messages"], option_text(ex["target"], ex["options"][ex["target"]]), ex["current"], ex["question"])[-260:]
              .replace("\n", " | "))


## Cell 42 — Positive and negative pairs

**What.** Classify every negative pair by **how hard** it is, and oversample the hardest kind.

| Negative type | Definition | Generic example |
|---|---|---|
| **semantically close** | the wrong agent is the other member of the domain's close pair | "it rejects my login" over VPN, scored against Network & VPN |
| **plausible but incorrect** | the conversation mentions the wrong agent's topic word (a distractor) | "Forget the printer. I need Photoshop installed." scored against Hardware |
| **obviously wrong** | nothing in the conversation points to that agent | a baggage question scored against Special Assistance |

**Why.** If most negatives are obvious, the head can get a low loss from easy topic matching and never learn close-pair boundaries. Close pairs are exactly where keyword routing fails, and in testing the banking close pair (Payments ↔ Card Support) is the hardest decision. Giving close negatives **3× weight** makes those mistakes cost more during training.

**Math.** Each pair gets a sample weight `w`, which enters the loss in Cell 43:

```text
w = 4   for positives               (balances 1 positive against 4 negatives)
w = 3   for semantically-close negatives
w = 1   for all other negatives
```

**Expect.** The counts of each negative type in the training pairs, and the total weight each group carries.

In [ ]:
# Cell 42 — Negative types and sample weights (close-pair negatives oversampled)

POS_WEIGHT, HARD_WEIGHT = 4.0, 3.0


def negative_type(pair, conv_text):
    if pair["label"] == 1:
        return "positive"
    if pair["question"] != ROUTE_Q:
        return "other question"
    info = generic_domains[pair["domain"]]
    if {pair["agent"], pair["target"]} == set(info["close_pair"]):
        return "semantically close"
    if any(k in conv_text for k in info["keywords"][pair["agent"]]):
        return "plausible but incorrect"
    return "obviously wrong"


conv_text = {ex["id"]: " ".join(m["content"] for m in ex["messages"]).lower() for ex in train_examples}
for p_ in generic_pairs:
    p_["kind"] = negative_type(p_, conv_text[p_["conv_id"]])
    p_["weight"] = {"positive": POS_WEIGHT, "semantically close": HARD_WEIGHT}.get(p_["kind"], 1.0)

kinds = Counter(p_["kind"] for p_ in generic_pairs)
print(f"{'pair type':26s} | {'count':>5} | {'weight each':>11} | {'total weight':>12}")
for kind in ["positive", "semantically close", "plausible but incorrect", "obviously wrong", "other question"]:
    w = {"positive": POS_WEIGHT, "semantically close": HARD_WEIGHT}.get(kind, 1.0)
    print(f"{kind:26s} | {kinds[kind]:5d} | {w:11.1f} | {kinds[kind] * w:12.1f}")

ex_hard = next((p_ for p_ in generic_pairs if p_["kind"] == "semantically close"), None)
if ex_hard:
    print(f"\nExample close negative: {ex_hard['conv_id']} (target {ex_hard['target']}) vs candidate {ex_hard['agent']}")

## Cell 43 — Loss

**What.** Two candidate losses. In the first full run the listwise loss won clearly (it took the top 5 configurations), so Phase 12's head selection now trains with the listwise loss.

**1. Weighted binary cross-entropy (pointwise).** Each (conversation, agent) pair answers one yes/no question: *is this agent appropriate?*

```text
loss = Σ w_n · BCE(sigmoid(score_n), label_n) / Σ w_n          (weights from Cell 42)
```

It matches the spec's labels exactly and works for any number of candidates. `BCEWithLogitsLoss` applies the sigmoid internally in a numerically stable way.

**2. Softmax over candidates (listwise).** Group a conversation's 5 pairs and apply cross-entropy over their 5 scores:

```text
loss = −log softmax(s_1 … s_5)[correct agent]
```

This optimizes exactly what routing uses at inference: *the correct agent must beat the other candidates of the same conversation*. Every batch must contain complete candidate groups, so we batch by conversation (`[conversations, 5, features]`).

**Expect.**
- Binary: near 0 for a confident correct score, large for a confident wrong one, 0.69 for a neutral score of 0.
- Listwise: small when the correct agent clearly wins the group, and log 5 ≈ 1.61 when all five scores are equal.

In [ ]:
# Cell 43 — Pointwise (weighted BCE) and listwise (softmax over the 5 candidates) losses

bce = nn.BCEWithLogitsLoss(reduction="none")


def weighted_bce(scores, labels, weights):
    return (bce(scores, labels) * weights).sum() / weights.sum()


def listwise_ce(scores, labels, mask=None):
    """scores, labels: [examples, K_MAX]; exactly one label is 1 per row. mask: which slots hold a real option
    (a yes / no question uses 2 of the 5 slots); padded slots get a score of -10,000, so softmax gives them ~0."""
    if mask is not None:
        scores = scores.masked_fill(~mask, -1e4)
    return F.cross_entropy(scores, labels.argmax(dim=1))


cases = [("confident & right (positive)", 6.0, 1.0), ("confident & right (negative)", -6.0, 0.0),
         ("confident & WRONG (positive)", -6.0, 1.0), ("neutral score 0", 0.0, 1.0)]
print("Binary (pointwise):")
print(f"  {'case':32s} | {'score':>6} | {'label':>5} | {'BCE':>7}")
for name, s_, y in cases:
    print(f"  {name:32s} | {s_:6.1f} | {y:5.0f} | {bce(torch.tensor([s_]), torch.tensor([y])).item():7.4f}")

labels = torch.tensor([[1.0, 0, 0, 0, 0]])
print("\nListwise (softmax over 5 candidates, correct = candidate 0):")
for name, s_ in [("correct clearly wins", [4.0, 0, 0, 0, 0]), ("all equal", [0.0] * 5), ("a wrong one wins", [0.0, 4, 0, 0, 0])]:
    print(f"  {name:22s} scores {s_} -> loss {listwise_ce(torch.tensor([s_]), labels).item():.4f}")

## Cell 44a — Freeze vs fine-tune, and the features the head sees

**First experiment:** ModernBERT **frozen**, compatibility head **trainable**. A frozen encoder is a fixed function, so every input's representation can be computed **once**, cached, and reused across all head experiments. That's what makes it possible to compare dozens of head designs.

**What changes when the encoder is fine-tuned?**
- Gradients must flow back through all the layers, so every training step needs a forward and a backward pass for every pair, in every epoch. Nothing can be cached.
- Memory grows with stored activations and optimizer state (about 1.8 GB for Adam on 149M parameters).
- The representations themselves adapt to routing. That's usually the biggest accuracy gain, at the risk of overfitting and forgetting general knowledge. Partial fine-tuning, or LoRA as in this repo's LoRA notebook, is a middle ground.

**Two families of frozen features per (conversation, agent) pair.**

| Family | How it's computed | What it captures |
|---|---|---|
| **cross** | one ModernBERT pass over `conversation [SEP] candidate` (the cross-encoder input). From that single output we average three **spans**: the whole conversation `c`, the **last user message** `l`, and the agent description `a` | conversation and description tokens attended to each other inside every layer |
| **dual** | three **separate** passes over the conversation alone, the last user message alone, and the description alone (each mean-pooled). Description vectors can be computed once per agent | each text understood on its own, as in a dual encoder |

**Why these matching features.** A head that sees the raw vectors learns *which domain* a conversation is from, and that doesn't transfer to unseen agents. We therefore give it **matching signals**: element-wise products and absolute differences between conversation-side and agent-side vectors, after standardizing each dimension.

```text
cross features = [ l⊙a,  c⊙a,  |l−a|,  |c−a| ]   (from the joint pass)         4 × 768 = 3,072
dual  features = [ c⊙a,  |c−a|,  l⊙a,  |l−a| ]   (from separate passes)        4 × 768 = 3,072
```

Keeping `l` (the last user message) separate gives the newest turn its own weight, which matters for topic changes.

**This cell** freezes the encoder and defines the feature functions. Cell 44b computes the features.

**Current agent.** Both passes include `current agent: <name>`: the joint text has it before the candidate, and the dual conversation text ends with it. `raw_features` also returns one explicit flag per pair, `is_current` (1 if the candidate **is** the current agent). It's appended to every feature set, so the head can learn a "stay" prior and when the text overrides it. The separate last-message pass is gone: alone, "ok" means nothing, while its span in the joint pass has read the history.

In [ ]:
# Cell 44a — Frozen encoder; functions for cross (span) and dual features, cached

for t in weights.values():
    t.requires_grad_(False)
print(f"trainable encoder parameters: {sum(t.numel() for t in weights.values() if t.requires_grad)}")

CROSS_CACHE, DUAL_CACHE = CACHE_DIR / f"span_features_q_{MODEL_TAG}.pt", CACHE_DIR / f"dual_features_q_{MODEL_TAG}.pt"
cross_cache = torch.load(CROSS_CACHE) if CROSS_CACHE.exists() else {}
dual_cache = torch.load(DUAL_CACHE) if DUAL_CACHE.exists() else {}


def cross_features(pairs):
    """One joint pass per (conversation, current agent, question, option) text -> [N, L, 5, H]:
    means over the conversation, its last message, the option, the whole input, and the question."""
    built = [build_text(p_["messages"], p_["current"], p_["candidate"], ENC["query"], question=p_["question"]) for p_ in pairs]
    ranges = [[(sp["messages"][0][0], sp["messages"][-1][1]), sp["messages"][-1], sp["candidate"], None, sp["question"]]
              for _, sp in built]
    return span_means([t for t, _ in built], ranges, cross_cache, CROSS_CACHE, "cross features")


def dual_vectors(texts):
    """Separate pass per distinct text, mean over all its tokens -> [N, L, H]."""
    return span_means(texts, [[None]] * len(texts), dual_cache, DUAL_CACHE, "dual features")[:, :, 0]


def turns_text(messages, current=None, question=None):
    """The conversation text alone (with its current agent and the question): the dual pass's conversation side."""
    return build_text(messages, current, None, ENC["query"], question=question)[0]


def raw_features(pairs):
    """-> cross [N, L, 5, H] (joint pass), dual [N, L, 2, H] (separate passes: conversation + question, option), is_current [N, 1].

    The last message gets no separate dual pass: alone, a reply like "ok" or "and the card?" has no meaning.
    In the joint pass its tokens attend to the whole history, so its span there keeps that meaning."""
    cross = cross_features(pairs)
    dual = torch.stack([dual_vectors([turns_text(p_["messages"], p_["current"], p_["question"]) for p_ in pairs]),
                        dual_vectors([ENC["passage"] + p_["candidate"] for p_ in pairs])], dim=2)
    is_current = torch.tensor([[float(p_["agent"] == p_["current"])] for p_ in pairs])
    return cross, dual, is_current

## Cell 44b — Compute the features for every generic pair

**What.** Run the feature functions from Cell 44a over all 17,710 generic (conversation, agent) pairs:
- one joint pass per pair (cross)
- one separate pass per distinct conversation (with its current agent) and agent description (dual)

**Caching.** Results are stored in `cache/` and keyed by the exact input text. A rerun of the notebook (or of this cell) reuses them and finishes in seconds.

**Shapes.** `cross` `[pairs, 4, H]` (conversation, last message, candidate, whole input). `dual` `[pairs, 2, H]` (conversation, candidate), and `is_current` `[pairs, 1]`.

**Expect.** About 22,600 encoder passes on the first run (about 3 minutes on a T4, much longer on CPU), and 0 new passes when cached.

In [ ]:
# Cell 44b — Compute (or load from cache) the features of every generic pair

t0 = time.perf_counter()
generic_cross, generic_dual, generic_current = raw_features(generic_pairs)
generic_cross, generic_dual = generic_cross.half(), generic_dual.half()      # half the RAM; layers go to the GPU one at a time
cross_cache.clear()                                                           # its contents are on disk and in generic_cross
log(f"generic features ready | {mem()}")
feature_time = time.perf_counter() - t0
print(f"cross {tuple(generic_cross.shape)} | dual {tuple(generic_dual.shape)}  (pairs, layers [last, mix], parts, H) | "
      f"is_current {tuple(generic_current.shape)}: {int(generic_current.sum())} pairs where the candidate is the current agent")
print(f"new encoder passes: {timing['encoder_passes']} in {timing['encoder_seconds']:.0f}s on {DEVICE} | feature step {feature_time:.0f}s")

## Cell 45a — Features with the question, and training over 2–5 options

**Features.** The joint (cross) pass now has a fifth span, the **question**. The head sees how the option relates to the conversation, to the last message **and to the question**:

```text
dual  : conversation + question  vs  option              -> products, |differences|       2H
cross : last message, conversation, QUESTION  vs  option -> products, |differences|       6H
sims  : 4 cosine similarities;  is_current: 1 only for the current agent of a routing question
```

**Training.** A yes / no question has 2 options and a routing question has 5, so every example sits on a grid of 5 slots with a **mask**. Padded slots get a score of −10,000, so the softmax gives them nothing. The loss is still listwise cross-entropy, over the real options only. The grid is kept in fp16 on the GPU and converted per batch.

**The head** is fixed: MLP-256 (Phase 7). The same head type is trained twice, once per group (Cell 45b).


In [ ]:
# Cell 45a — Matching features (with the question), the head, and masked listwise training on a [examples, 5, F] grid

import copy
import statistics

# One fixed configuration (Phase 7): both passes + similarities, MLP-256 head
FEATURE_SETS = ["dual + cross + sim"]
HEADS = ["mlp-256"]
TRAINED = [(fs, layer, head) for fs in FEATURE_SETS for layer in LAYERS for head in HEADS]
CONFIGS = TRAINED
LR, WEIGHT_DECAY = 3e-4, 1e-2
MAX_EPOCHS = int(os.environ.get("ROUTER_MAX_EPOCHS", "400"))   # lower it only for quick smoke tests
PATIENCE = 40


def feature_stats(cross, dual):
    return (cross.mean(0, keepdim=True), cross.std(0, keepdim=True) + 1e-6,
            dual.mean(0, keepdim=True), dual.std(0, keepdim=True) + 1e-6)


def matching_features(cross, dual, stats):
    """Standardize, then products and absolute differences against the option (a):
    dual  [conversation + question vs option]                               -> 2H
    cross [last message, conversation, QUESTION, each vs the option]        -> 6H
    -> matching [N, 8H] and similarities [N, 4] (dual, conversation, last message, question vs option)."""
    cm, cs, dm, ds = stats
    zc, zd = (cross - cm) / cs, (dual - dm) / ds
    c, a = zd[:, 0], zd[:, 1]
    dual_f = torch.cat([c * a, (c - a).abs()], dim=1)
    sims = [F.cosine_similarity(c, a, dim=-1)]
    c, l, a, q = zc[:, 0], zc[:, 1], zc[:, 2], zc[:, 4]
    cross_f = torch.cat([l * a, c * a, q * a, (l - a).abs(), (c - a).abs(), (q - a).abs()], dim=1)
    sims += [F.cosine_similarity(c, a, dim=-1), F.cosine_similarity(l, a, dim=-1), F.cosine_similarity(q, a, dim=-1)]
    return torch.cat([dual_f, cross_f], dim=1), torch.stack(sims, dim=1)


def features(match, sims, is_current, feature_set=None):
    """Matching features + similarities + the is_current flag (1 only for the current agent of a routing question)."""
    return torch.cat([match, sims, is_current], dim=1)


class ResidualHead(nn.Module):
    """Dropout -> Linear(F -> 256) -> LayerNorm -> GELU, then a residual block 256 -> 256, then -> 1 score."""

    def __init__(self, n_in, d=256, drop=0.3):
        super().__init__()
        self.inp = nn.Sequential(nn.Dropout(0.2), nn.Linear(n_in, d), nn.LayerNorm(d), nn.GELU())
        self.block = nn.Sequential(nn.Dropout(drop), nn.Linear(d, d), nn.LayerNorm(d), nn.GELU())
        self.out = nn.Linear(d, 1)

    def forward(self, x):
        h = self.inp(x)
        return self.out(h + self.block(h))


def make_head(kind, n_in):
    if kind == "linear":
        return nn.Linear(n_in, 1)
    if kind == "resid-ls":
        return ResidualHead(n_in)
    hidden, drop = {"mlp-64": (64, 0.5), "mlp-256": (256, 0.3)}[kind]
    return nn.Sequential(nn.Dropout(0.2), nn.Linear(n_in, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, 1))


def masked_scores(head, X, mask):
    """X [B, K_MAX, F] (fp16 storage) -> scores [B, K_MAX], padded slots at -10,000."""
    return head(X.float()).squeeze(-1).masked_fill(~mask, -1e4)


def eval_loss(head, X, y, m, idx):
    with torch.no_grad():
        total = sum(F.cross_entropy(masked_scores(head, X[b], m[b]), y[b].argmax(1), reduction="sum").item() for b in idx.split(2048))
    return total / max(1, len(idx))


def train_head(head_kind, X, y, m, idx_tr, idx_va=None, epochs=MAX_EPOCHS, patience=PATIENCE, seed=42):
    """X [examples, K_MAX, F] fp16 on DEVICE, y labels [examples, K_MAX], m mask [examples, K_MAX];
    idx_tr / idx_va: example indices. Masked listwise loss; early stopping on idx_va."""
    torch.manual_seed(seed)
    head = make_head(head_kind, X.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    g = torch.Generator().manual_seed(seed)
    idx_tr = idx_tr.to(DEVICE)
    idx_va = None if idx_va is None else idx_va.to(DEVICE)
    best, history = (float("inf"), None, 0), []
    for epoch in range(1, epochs + 1):
        head.train()
        for b in idx_tr[torch.randperm(len(idx_tr), generator=g).to(DEVICE)].split(64):
            loss = F.cross_entropy(masked_scores(head, X[b], m[b]), y[b].argmax(1))
            opt.zero_grad(); loss.backward(); opt.step()
        if idx_va is None:
            continue
        head.eval()
        history.append((eval_loss(head, X, y, m, idx_tr[:20000]), eval_loss(head, X, y, m, idx_va)))
        if history[-1][1] < best[0]:
            best = (history[-1][1], copy.deepcopy(head.state_dict()), epoch)
        if epoch - best[2] >= patience:
            break
    if idx_va is not None:
        head.load_state_dict(best[1])
    head.eval()
    return head, best[2], history


def heldout_probs(head, X, m, idx):
    with torch.no_grad():
        return torch.cat([torch.softmax(masked_scores(head, X[b], m[b]), dim=1) for b in idx.to(DEVICE).split(2048)])


print(f"head configuration: {CONFIGS[0]} (fixed), one per group {GROUPS}; masked listwise loss over 2-{K_MAX} options, "
      f"lr {LR}, weight decay {WEIGHT_DECAY}")


## Cell 45b — Each head scored on its own data

**What.** For each head separately: the 5 leave-domains-out folds of its own group. The 3 training folds fit the standardization and the head, the next fold decides early stopping, and the held-out fold is scored.

**Expect.** Route head: held-out routing on the same 6,850 conversations as earlier runs (run 16: 5,900). Choice head: held-out yes / no, choice and score, where each held-out fold is a set of question families the head never saw (run 16: 442 / 843, 241 / 452, 104 / 229).


In [ ]:
# Cell 45b — Each head on its own data: 5 leave-domains-out folds within the group, held-out score per task

H_ = generic_cross.shape[-1]
TASKS = {"route": ["routing"], "choice": ["noul", "choice", "score"]}


def group_data(g):
    """Everything one head trains and is scored on: its examples, pairs, option grid, labels, mask and folds."""
    ex_ids = [i for i, gg in enumerate(ex_group) if gg == g]
    local = {e: k for k, e in enumerate(ex_ids)}
    pair_ids = [j for j, p_ in enumerate(generic_pairs) if ex_group[p_["ex"]] == g]
    pex = torch.tensor([local[generic_pairs[j]["ex"]] for j in pair_ids])
    pslot = torch.tensor([generic_pairs[j]["slot"] for j in pair_ids])
    n = len(ex_ids)
    mask = torch.zeros(n, K_MAX, dtype=torch.bool)
    mask[pex, pslot] = True
    labels = torch.zeros(n, K_MAX)
    labels[pex, pslot] = torch.tensor([generic_pairs[j]["label"] for j in pair_ids])
    assert (labels.sum(1) == 1).all(), f"{g}: every example has exactly one right option"
    fold = torch.tensor([next(f for f, doms in enumerate(FOLDS[g]) if conv_domain[e] in doms) for e in ex_ids])
    return {"n": n, "pair_ids": torch.tensor(pair_ids), "pex": pex, "pslot": pslot, "mask": mask.to(DEVICE),
            "labels": labels.to(DEVICE), "fold": fold, "tasks": [ex_task[e] for e in ex_ids]}


def group_stats(gd, li, pair_sel=None):
    """Standardization statistics from this group's pairs only (optionally a subset of them)."""
    ids = gd["pair_ids"] if pair_sel is None else gd["pair_ids"][pair_sel]
    return feature_stats(generic_cross[ids, li].float(), generic_dual[ids, li].float())


def grid_features(gd, li, stats, chunk=8192):
    """This group's pair features on its [examples, K_MAX, F] grid, fp16 on the GPU (padded slots stay 0)."""
    X = torch.zeros(gd["n"], K_MAX, 8 * H_ + 4 + 1, dtype=torch.float16, device=DEVICE)
    st = tuple(s_.to(DEVICE) for s_ in stats)
    for s0 in range(0, len(gd["pair_ids"]), chunk):
        ids = gd["pair_ids"][s0:s0 + chunk]
        match, sims = matching_features(generic_cross[ids, li].to(DEVICE).float(), generic_dual[ids, li].to(DEVICE).float(), st)
        X[gd["pex"][s0:s0 + chunk].to(DEVICE), gd["pslot"][s0:s0 + chunk].to(DEVICE)] = \
            features(match, sims, generic_current[ids].to(DEVICE)).half()
        del match, sims
    return X


GD = {g: group_data(g) for g in GROUPS}
results = {}
t0 = time.perf_counter()
for g in GROUPS:
    gd = GD[g]
    r = results[g] = {"best_epochs": [], "histories": [], "by_task": {t: [0, 0] for t in TASKS[g]}}
    log(f"{g} head: {N_FOLDS} folds | {gd['n']} examples, {len(gd['pair_ids'])} pairs")
    for f in range(N_FOLDS):
        te, va = gd["fold"] == f, gd["fold"] == (f + 1) % N_FOLDS
        tr = ~(te | va)
        for li, layer in enumerate(LAYERS):
            X = grid_features(gd, li, group_stats(gd, li, tr[gd["pex"]]))
            for fs, head_kind in [(fs, h) for fs in FEATURE_SETS for h in HEADS]:
                te_idx = te.nonzero().flatten()
                head, best_epoch, history = train_head(head_kind, X, gd["labels"], gd["mask"], tr.nonzero().flatten(), va.nonzero().flatten())
                probs = heldout_probs(head, X, gd["mask"], te_idx).cpu()
                ok = (gd["labels"].cpu()[te_idx, probs.argmax(1)] == 1)
                r["best_epochs"].append(best_epoch); r["histories"].append(history)
                for i, good in zip(te_idx.tolist(), ok.tolist()):
                    r["by_task"][gd["tasks"][i]][0] += good; r["by_task"][gd["tasks"][i]][1] += 1
                log(f"  {g} fold {f} | best epoch {best_epoch:3d} | held-out {int(ok.sum())}/{len(te_idx)} | {time.perf_counter() - t0:.0f}s")
                del head
            del X
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
lodo_time = time.perf_counter() - t0

BEST_CFG = CONFIGS[0]
print(f"\n{BEST_CFG} | {lodo_time:.0f}s | held-out domains, each head on its own data:")
for g in GROUPS:
    print(f"  {g} head (best epoch per fold {results[g]['best_epochs']}):")
    for t, (ok, n) in results[g]["by_task"].items():
        print(f"    {t:8s} {ok:>6} / {n:<6} ({ok / max(1, n):5.1%})")
lodo_best = tuple(results["route"]["by_task"]["routing"])               # comparable with earlier E1 (the same 6,850)
print(f"\nrouter encoder: {ROUTER_ENCODER} | routing held-out {lodo_best[0]}/{lodo_best[1]}")


## Cell 45c — Loss curves of the winner, and the final head(s)

**What.**
1. Plot the training and early-stopping losses for all 5 folds. This is the loss-decay check: how long does loss on **unseen** domains keep falling?
2. Train the **final head** on **all generic domains**. Standardization is refit on all generic data. Each head trains for the median best epoch from its folds, since there's no held-out data left for early stopping.
3. Define `score_candidates(messages, agents, current=None)`, the finished router used by every later cell. It scores any set of candidate `{name: description}` agents for a conversation, given the agent currently holding it. The scores are log-probabilities; a softmax over them gives back the probabilities exactly.

**Expect.** Five pairs of curves, and the final head trained in seconds.

In [ ]:
# Cell 45c — Loss decay, the two final heads (each on all of its own data), and the router / ask functions

fs, best_layer, head_kind = BEST_CFG
BEST_LI = LAYERS.index(best_layer)

fig, axes = plt.subplots(1, len(GROUPS), figsize=(9 * len(GROUPS), 4), squeeze=False)
for ax, g in zip(axes[0], GROUPS):
    for f, hist in enumerate(results[g]["histories"]):
        ax.plot([h[0] for h in hist], color="tab:blue", alpha=0.5, label="training domains" if f == 0 else None)
        ax.plot([h[1] for h in hist], color="tab:orange", alpha=0.8, label="unseen early-stopping domains" if f == 0 else None)
    ax.set_xlabel("epoch"); ax.set_ylabel("masked listwise loss"); ax.set_title(f"{g} head — loss decay per fold")
    ax.legend(); ax.grid(alpha=0.3)
plt.show()


def train_final(g):
    """The group's head on ALL of its own examples, for the median best epoch of its folds, with its own statistics."""
    gd = GD[g]
    stats = tuple(s_.cpu() for s_ in group_stats(gd, BEST_LI))
    epochs = max(1, int(statistics.median(results[g]["best_epochs"])))
    X_all = grid_features(gd, BEST_LI, stats)
    head, _, _ = train_head(head_kind, X_all, gd["labels"], gd["mask"], torch.arange(gd["n"]), epochs=epochs)
    del X_all
    return {"head": head, "layer": BEST_LI, "stats": stats, "epochs": epochs}


log(f"training the final heads {GROUPS}")
t0 = time.perf_counter()
FINAL = {g: train_final(g) for g in GROUPS}
final_heads = {"route": FINAL["route"]}                          # the router (Cells 46-71) uses ONLY the route head
choice_heads = {"choice": FINAL["choice"]} if "choice" in FINAL else final_heads
final_epochs = {g: h["epochs"] for g, h in FINAL.items()}
head_train_time = time.perf_counter() - t0
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f"final heads {final_epochs} (epochs), {head_train_time:.1f}s | {mem()}")


def head_probs(heads, cross, dual, is_current):
    """Probabilities over the options of ONE question, from the given head(s), each with its own statistics -> [N]."""
    total = 0
    with torch.no_grad():
        for h in heads.values():
            match, sims = matching_features(cross[:, h["layer"]], dual[:, h["layer"]], h["stats"])
            X = features(match.to(DEVICE), sims.to(DEVICE), is_current.to(DEVICE))
            total = total + torch.softmax(h["head"](X).squeeze(-1), dim=0)
    return total / len(heads)


def candidate_features(messages, options, current=None, question=ROUTE_Q):
    """Encode one question: raw cross / dual features and is_current for every option {name: description}."""
    rows = [{"messages": messages, "current": current, "question": question, "agent": n, "candidate": option_text(n, d)}
            for n, d in options.items()]
    return raw_features(rows)


def score_candidates(messages, agents, current=None, question=ROUTE_Q):
    """The ROUTER (route head): log-probabilities over the agents -> [len(agents)]."""
    return head_probs(final_heads, *candidate_features(messages, agents, current, question)).log().cpu()


def ask(messages, question, options, current=None):
    """A general choice question (CHOICE head) -> {option: probability}. options: list of names or {name: description}."""
    opts = options if isinstance(options, dict) else {o: "" for o in options}
    p = head_probs(choice_heads, *candidate_features(messages, opts, current, question)).cpu()
    return dict(zip(opts, p.tolist()))


demo = [{"role": "user", "content": "Help! My payouts have been failing for 3 days!"}]
print("\nthe decider's model-card example, asked to the choice head:")
for q, opts in [("Which team should handle this?", ["billing", "sales", "retail"]), ("Does this convey urgency?", ["yes", "no"]),
                ("How frustrated is the writer?", ["calm", "frustrated", "furious"])]:
    print(f"  {q:34s} " + " | ".join(f"{k} {v:.2f}" for k, v in ask(demo, q, opts).items()))


## Cell 46 — CPU performance

**What.** Measure what this design costs on this CPU:

1. one ModernBERT pass (feature extraction)
2. training the head
3. one **live routing decision** for a brand-new conversation (5 encoder passes, nothing cached)
4. a projection of what **fine-tuning** the encoder would cost here

**Why.** The notebook must stay usable on the provided CPU-only environment, and the numbers show why the frozen-encoder design was chosen first.

**Math.**

```text
routing latency ≈ 5 × (one encoder pass)          five candidates, cross-encoder
fine-tune epoch ≈ N_pairs × 3 × (one encoder pass) forward + backward, nothing cacheable
```

**Expect.**
- An encoder pass takes on the order of 100 ms for these short inputs, so a routing decision (5 passes, batched) takes well under a second.
- Head training takes seconds to a minute or so, even over thousands of epochs, because every epoch reuses the cached features.
- A full fine-tuning epoch would cost minutes of compute per epoch, repeated every epoch with nothing cacheable. Memory is the bigger obstacle: gradients plus Adam's two moment estimates for 149M parameters need about 1.8 GB on top of the weights and activations, and this 8 GB machine has well under 1 GB free. That's why we freeze the encoder here.

**Note.** With features cached, every head experiment only costs head training. The encoder passes are paid once.

In [ ]:
# Cell 46 — Timings for the frozen-encoder router

fresh = [{"role": "user", "content": "Hi, my salary transfer bounced back, what should I do now?"}]
t0 = time.perf_counter()
fresh_scores = score_candidates(fresh, AGENTS)               # live passes for the new conversation (descriptions are cached)
live_latency = time.perf_counter() - t0
fresh_choice = AGENT_NAMES[int(torch.argmax(fresh_scores))]

per_pass = (timing["encoder_seconds"] / timing["encoder_passes"]) if timing["encoder_passes"] else float("nan")
print(f"{'measurement':52s} | value")
print("-" * 80)
print(f"{'device':52s} | {DEVICE}")
print(f"{'mean encoder pass during feature extraction':52s} | {per_pass * 1000:.1f} ms per text")
print(f"{'leave-domains-out selection (all configs x 5 folds)':52s} | {lodo_time:.0f} s")
print(f"{'final head training':52s} | {head_train_time:.1f} s")
print(f"{'live routing decision for a new conversation':52s} | {live_latency:.2f} s  -> {fresh_choice}")

---
# Phase 13 — Routing evaluation

## Cell 47a — Basic accuracy: routing and metrics

**What.** Route every **banking** conversation, a domain the head has never seen, and count

```text
routing accuracy = correct routes / total conversations
```

We report on two banking test sets: the **benchmark** (50 held-out conversations) and **all banking** (benchmark + 175 extended = 225).

**Why.** A single accuracy number hides where routing fails. The spec asks for each of these separately:

| Metric | Definition |
|---|---|
| overall routing accuracy | correct agent ranked first among the 5 |
| obvious-case accuracy | `obvious` conversations (single clear request) |
| hard-negative accuracy | for Payments / Card targets: correct agent scores **above its close sibling** |
| closely-related-agent accuracy | 5-way accuracy on Payments / Card targets |
| handoff accuracy | conversations containing an explicit handoff |
| "OK"-after-handoff accuracy | handoff conversations whose final user message is just "OK" / "okay" / "sure" |
| ding-dong routing accuracy | final routing decision on ding-dong conversations |
| context-dependent accuracy | the last message alone is ambiguous |

**This cell** defines the routing loop and the metrics. Cell 47b runs them on banking.

In [ ]:
# Cell 47a — Routing and the category metrics

def route_conversations(conversations, agents=AGENTS):
    results = []
    for ex in conversations:
        s_ = score_candidates(ex["messages"], agents, ex["current_agent"])
        results.append({"ex": ex, "scores": s_, "pred": list(agents)[int(torch.argmax(s_))]})
    return results


def sibling(name):
    return next(iter(CLOSE_PAIR - {name}))


def category_metrics(results):
    def count(rows):
        return (sum(r["pred"] == r["ex"]["target_agent"] for r in rows), len(rows))
    def tagged(tag):
        return [r for r in results if tag in r["ex"]["tags"]]
    close = [r for r in results if r["ex"]["target_agent"] in CLOSE_PAIR]
    hard = sum(r["scores"][AGENT_NAMES.index(r["ex"]["target_agent"])] > r["scores"][AGENT_NAMES.index(sibling(r["ex"]["target_agent"]))] for r in close)
    return {
        "overall routing accuracy": count(results),
        "obvious-case accuracy": count(tagged("obvious")),
        "hard-negative accuracy": (int(hard), len(close)),
        "closely-related-agent accuracy": count(close),
        "handoff accuracy": count(tagged("handoff")),
        "OK-after-handoff accuracy": count(tagged("ok_after_handoff")),
        "ding-dong routing accuracy": count(tagged("ding_dong")),
        "context-dependent accuracy": count(tagged("context")),
        "short (1-3 messages)": count(tagged("short")),
        "medium (4-8 messages)": count(tagged("medium")),
        "long (9-15 messages)": count(tagged("long")),
    }




def fmt(pc):
    passed, total = pc
    return f"{passed:>3} pass / {total - passed:>3} fail  ({passed / total:5.1%})" if total else "n/a"

## Cell 47b — Banking test results

**What.** Route every banking conversation with the final router from Cell 45c and print **pass / fail** per category for the benchmark (50) and for all banking conversations (225).

The final router is the configuration that transferred best to unseen generic domains, trained on all 27 of them. It has never seen a banking conversation or a banking agent.

**Expect.** One row per metric for both test sets.

In [ ]:
# Cell 47b — Banking TEST: pass / fail per category (the head was trained on generic domains only)

log("routing the banking test conversations")
eval_results = route_conversations(banking_benchmark)
all_results = route_conversations(banking_all)
log(f"banking routing done: {len(all_results)} conversations")
eval_metrics, all_metrics = category_metrics(eval_results), category_metrics(all_results)

print(f"{'metric':32s} | {'benchmark (50)':>28} | {'all banking (225)':>28}")
print("-" * 96)
for k in eval_metrics:
    print(f"{k:32s} | {fmt(eval_metrics[k]):>28} | {fmt(all_metrics[k]):>28}")

## Cell 48 — Confusion matrix

**What.** Cross-tabulate the correct agent (rows) against the selected agent (columns) over **all 225 banking test conversations**, then pull out the **Payments ↔ Card Support** block on its own.

**Why.** Accuracy tells us *how often* the router is wrong. The confusion matrix tells us *how*: which agent absorbs which errors. If errors concentrate in the Payments / Card block, the router has learned the topics but not the boundary. If they spread into General Support, it's uncertain and falling back to the generic agent.

**Shapes.** A `[5, 5]` count matrix whose rows sum to the number of test conversations per correct agent. The close pair is a `[2, 2]` sub-matrix.

**Expect.** Most counts on the diagonal. The error list shows the misrouted benchmark conversations.

In [ ]:
# Cell 48 — Confusion matrix on all banking test conversations: rows = correct agent, columns = selected agent

short_names = {"Payments Agent": "Payments", "Card Support Agent": "Card", "Investment Agent": "Investment",
               "Insurance Agent": "Insurance", "General Customer Support Agent": "General"}
idx = {n: i for i, n in enumerate(AGENT_NAMES)}
confusion = torch.zeros(5, 5, dtype=torch.long)
for r in all_results:
    confusion[idx[r["ex"]["target_agent"]], idx[r["pred"]]] += 1

labels5 = [short_names[n] for n in AGENT_NAMES]
print(f"{'correct \\ selected':>18}" + "".join(f"{l:>11}" for l in labels5) + f"{'total':>8}")
for i, l in enumerate(labels5):
    print(f"{l:>18}" + "".join(f"{v:>11d}" for v in confusion[i].tolist()) + f"{int(confusion[i].sum()):>8}")
assert confusion.sum() == len(all_results)
print(f"correct on the diagonal: {int(confusion.diag().sum())} / {len(all_results)}")

pc = [idx["Payments Agent"], idx["Card Support Agent"]]
print("\nPayments <-> Card Support block (rows correct, columns selected):")
print(confusion[pc][:, pc])

errors = [r for r in eval_results if r["pred"] != r["ex"]["target_agent"]]
print(f"\nBenchmark errors ({len(errors)} of {len(eval_results)}):")
for r in errors:
    print(f"  {r['ex']['id']:7s} correct {short_names[r['ex']['target_agent']]:10s} -> selected {short_names[r['pred']]:10s} "
          f"| last user message: {r['ex']['messages'][-1]['content']!r}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(confusion.numpy(), cmap="Blues")
for i in range(5):
    for j in range(5):
        ax.text(j, i, int(confusion[i, j]), ha="center", va="center")
ax.set_xticks(range(5), labels5, rotation=30)
ax.set_yticks(range(5), labels5)
ax.set_xlabel("selected agent")
ax.set_ylabel("correct agent")
ax.set_title("Banking test confusion matrix (225 conversations)")
plt.tight_layout()
plt.show()

---
# Phase 14 — Ding-dong routing

## Cell 50 — Define the ding-dong problem

**What.** A "ding-dong" conversation is one where the user repeatedly jumps between agents' topics:

```text
User → investment topic      → Investment
User → payment topic         → Payments
User → investment topic      → Investment
…
```

Sometimes the jumps are legitimate (a real second question). Sometimes they're deliberate or playful ("lol no, block my card", "just kidding").

**Why.** A router that re-decides after every user turn can flip-flop: it might switch on a stray keyword, or fail to follow a real change of topic. To study this, we need the **expected agent after every user turn**, not only at the end. Each sequence in `data/modernbert_router_dingdong.json` annotates every user turn with:

| `turn_type` | Meaning |
|---|---|
| `start` | first request |
| `same_topic` | follow-up on the current topic: the agent should **not** change |
| `topic_change` | a genuine new request: switching is correct |
| `ding_dong` | the user deliberately bounces to another agent's topic: following them is correct, but it's a *bounce* |
| `ambiguous` | the request could belong to more than one agent; history decides |

**Shapes.** 5 sequences of 9 messages each, with 5 annotated user turns per sequence (25 routing decisions).

**Expect.** One sequence printed with its per-turn annotations, and a summary of the turn types.

In [ ]:
# Cell 50 — Ding-dong sequences with the expected agent after every user turn

for seq in router_dingdong:
    print(f"{seq['id']}: {seq['title']}")

seq = router_dingdong[0]
print(f"\n{seq['id']} in full:")
ann = {u["message_index"]: u for u in seq["user_turns"]}
for i, m in enumerate(seq["messages"]):
    note = f"   -> expect {short_names[ann[i]['expected_agent']]} ({ann[i]['turn_type']})" if i in ann else ""
    print(f"  {m['role']:9s}: {m['content']}{note}")

print("\nturn types:", dict(Counter(u["turn_type"] for s_ in router_dingdong for u in s_["user_turns"])))

## Cell 51 — Evaluate routing stability

**What.** Replay each ding-dong conversation and route **after every user turn**, using the conversation history up to and including that turn. Record:

```text
Turn | user message | expected agent | selected agent | probability | switch?
```

**Why.** This is how a router behaves inside a live multi-agent system: it's consulted again on every user message. A decision is a **switch** if the selected agent differs from the one selected at the previous user turn.

**Math.** For user turn `t`, `prefix_t = messages[0 : index_t + 1]`, so the prefix always ends with that user message, and `selected_t = argmax_k score(prefix_t, agent_k)`.

**Shapes.** 25 prefixes × 5 candidates = 125 encoder passes (cached after the first run).

**Expect.** A per-turn table for each sequence, and per-turn accuracy overall.

**Current agent.** This is a live simulation. The agent holding the conversation at each turn is the router's **own previous pick** (none at the first turn), exactly as in a deployed system. A wrong pick can therefore carry over to the next turn.

In [ ]:
# Cell 51 — Route after every user turn and mark switches

def replay(seq):
    """Live simulation: the current agent is the router's OWN previous pick (so an early mistake can carry over)."""
    rows, prev = [], None
    for u in seq["user_turns"]:
        prefix = seq["messages"][:u["message_index"] + 1]
        s_ = score_candidates(prefix, AGENTS, prev)
        pr = torch.softmax(s_, dim=0)
        sel = AGENT_NAMES[int(torch.argmax(s_))]
        rows.append({"turn": len(rows) + 1, "message": prefix[-1]["content"], "expected": u["expected_agent"],
                     "type": u["turn_type"], "selected": sel, "prob": pr.max().item(),
                     "switch": prev is not None and sel != prev})
        prev = sel
    return rows


dingdong_rows = {seq["id"]: replay(seq) for seq in router_dingdong}

for sid, rows in dingdong_rows.items():
    print(f"\n{sid}: {next(s_['title'] for s_ in router_dingdong if s_['id'] == sid)}")
    print(f"  {'turn':>4} | {'user message':44s} | {'expected':10s} | {'selected':10s} | {'prob':>5} | switch?")
    for r in rows:
        ok = "" if r["selected"] == r["expected"] else "  x"
        msg = r["message"] if len(r["message"]) <= 44 else r["message"][:41] + "..."
        print(f"  {r['turn']:>4} | {msg:44s} | {short_names[r['expected']]:10s} | {short_names[r['selected']]:10s} | "
              f"{r['prob']:5.2f} | {'SWITCH' if r['switch'] else ''}{ok}")

all_rows = [r for rows in dingdong_rows.values() for r in rows]
turn_accuracy = sum(r["selected"] == r["expected"] for r in all_rows) / len(all_rows)
print(f"\nper-turn routing accuracy over {len(all_rows)} user turns: {turn_accuracy:.1%}")

---
# Phase 15 — Final analysis

## Cell 58 — Final summary

**The journey.**

```text
SmolLM2
  ↓
causal generation                    token i sees tokens ≤ i, predicts token i+1

ModernBERT
  ↓
bidirectional representation         every token sees every token; RoPE for position;
                                     local (±64) and global layers; GeGLU MLP; pre-norm residuals
                                     → [B, S, 1024], built by hand in 01 from the real checkpoint
                                       and verified against the reference implementation

Compatibility head
  ↓
dynamic choice scoring               mean-pooled (conversation ⊕ candidate) → Linear(1024 → 1) → score

Multi-agent router
  ↓
conversation + candidate agent       5 cross-encoder passes → softmax → selected agent
  ↓
score
```

**The encoder.** `01_modernbert_by_hand.ipynb` built ModernBERT's forward pass by hand from the real ModernBERT-large checkpoint (28 layers, H = 1,024) and matched the reference `last_hidden_state` to about 3e-5. This notebook runs the same checkpoint through the library.

**How the router was trained and tested.** The compatibility head was trained **only on generic conversations from non-banking domains**, and its configuration was chosen by leaving whole domains out. It was then tested on **banking**, whose five agents it had never seen. Every banking number below is therefore a test of compatibility between a conversation and an *unseen* agent description.

**The question we set out to answer:** *can a ModernBERT cross-encoder learn the compatibility between a conversation and a dynamically supplied agent description, and use it to route a multi-agent conversation?* The table below gives the measured answer in absolute numbers.

In [ ]:
# Cell 58 — Final summary in absolute numbers

def row(name, test_set, passed, total):
    print(f"{name:52s} | {test_set:34s} | {total:>7} | {passed:>4} | {total - passed:>4}")


print(f"{'approach / measurement':52s} | {'test set':34s} | {'samples':>7} | {'pass':>4} | {'fail':>4}")
print("-" * 112)
row("leave-domains-out (router head)", f"generic, all {len(generic_domains)} domains held out", *lodo_best)
row("5-agent router", "banking benchmark", *eval_metrics["overall routing accuracy"])
row("5-agent router", "all banking conversations", *all_metrics["overall routing accuracy"])
row("5-agent router", "banking ding-dong, per user turn", int(round(turn_accuracy * len(all_rows))), len(all_rows))
row("5-agent router", "banking ding-dong, full conversation",
    sum(rows[-1]["selected"] == rows[-1]["expected"] for rows in dingdong_rows.values()), len(dingdong_rows))

print(f"winning head: {BEST_CFG} | epochs {final_epochs}, {head_train_time:.1f}s | device {DEVICE} | live routing decision: {live_latency:.2f}s")

## Cell 59 — Try it on five brand-new conversations

**What.** Five conversations, each with 8 user turns (40 in total), from five companies: an online shop, a home internet and mobile provider, a furniture store, an energy supplier and an online course provider. None of their agents appears in the training data. (The training data does have a mobile-network domain, but with different, narrower agents.) Like real customer service, each company has just **three broad agents**, for example Orders, Returns & Refunds and Payments. That's far fewer than the five narrow, partly overlapping agents of the training domains.

**Live, with a current agent.** The router is called after **every** user turn with the whole history, assistant replies included. The agent holding the conversation is the router's **own previous pick** (none at the first turn), exactly as it would be deployed. The table prints that current agent and the assistant lines, so you can see everything the router read.

For every turn the cell prints the router's pick, its probability and the probability of the **expected** agent. All the numbers are kept in `turn_log`.

**What the conversations exercise.**
- **Clear requests:** "Where is my order?", "My internet has been down since this morning."
- **Topic switches mid-conversation:** order → double charge → return → address change.
- **Short replies that only make sense from the history and the current agent:** "55102", "afternoon", "12 Harbour Road", "Chrome" and "yes" as answers to the agent's question, and "ok" or "ok thanks" after an answer.

**Expect.** Clear first messages and explicit switches should route correctly. The short replies test the current-agent signal: each should stay with the agent who just asked or answered.

**How to use it.** Edit `NEW_TESTS` to try your own agents and conversations. No retraining is needed, because agents are just names and descriptions.


In [ ]:
# Cell 59 — Five new conversations from domains the head never saw, routed live

# Each company has a few broad agents, as real customer service does; then (user message, expected agent, assistant reply) per turn.
NEW_TESTS = [
    {"domain": "online shop",
     "agents": {
         "Orders Agent": "Handles orders: placing and changing orders, delivery addresses, tracking and late deliveries.",
         "Returns & Refunds Agent": "Handles returns: sending items back, return labels and getting money back for returned items.",
         "Payments Agent": "Handles payments: paying, failed or double charges, saved cards and invoices."},
     "turns": [
         ("Where is my order? It was supposed to arrive yesterday.", "Orders Agent", "Can you share the order number?"),
         ("55102", "Orders Agent", "It's with the courier and arrives today."),
         ("Also I was charged twice for it.", "Payments Agent", "I see two charges; the extra one will be reversed within 3 days."),
         ("ok thanks", "Payments Agent", "Anything else I can help with?"),
         ("Yes, the shoes in that order are too small. Can I send them back?", "Returns & Refunds Agent", "Sure, I've emailed you a return label."),
         ("how long until I get my money back?", "Returns & Refunds Agent", "5 to 7 days after we receive them."),
         ("Can I change the delivery address for my other order?", "Orders Agent", "What's the new address?"),
         ("5 Mill Lane", "Orders Agent", None)]},
    {"domain": "home internet and mobile provider",
     "agents": {
         "Technical Support Agent": "Fixes problems: internet down or slow, router issues, no signal and connection faults.",
         "Bills & Payments Agent": "Handles bills: charges, higher bills, payment plans and paying a bill.",
         "Plans & Upgrades Agent": "Handles plans: changing or upgrading a plan, faster speeds and phone upgrades."},
     "turns": [
         ("My internet has been down since this morning.", "Technical Support Agent", "Have you restarted the router?"),
         ("yes twice", "Technical Support Agent", "I've reset the line from our side; please try now."),
         ("It works now. Why was my bill higher this month?", "Bills & Payments Agent", "There was a roaming charge from your trip."),
         ("ah right, can I pay it in two parts?", "Bills & Payments Agent", "Yes, I've split it into two payments."),
         ("I'd like a faster internet plan.", "Plans & Upgrades Agent", "The 500 Mbps plan is 10 more a month. Want it?"),
         ("yes please", "Plans & Upgrades Agent", "Done, it starts next month."),
         ("Does the new plan come with a phone upgrade?", "Plans & Upgrades Agent", "Yes, you can pick a new phone in the app."),
         ("The router light is blinking red again.", "Technical Support Agent", None)]},
    {"domain": "furniture store",
     "agents": {
         "Orders & Delivery Agent": "Handles furniture orders: placing orders, delivery dates and tracking deliveries.",
         "Assembly Service Agent": "Books our fitters to put furniture together at your home.",
         "Returns Agent": "Handles returns of furniture and damaged items, and refunds for them."},
     "turns": [
         ("When will my sofa be delivered?", "Orders & Delivery Agent", "It's scheduled for Thursday. Morning or afternoon?"),
         ("afternoon", "Orders & Delivery Agent", "Booked for Thursday afternoon."),
         ("Can someone put the wardrobe together for me too?", "Assembly Service Agent", "A fitter can come on Friday. Shall I book it?"),
         ("yes", "Assembly Service Agent", "Booked for Friday."),
         ("The coffee table arrived with a scratch, I want to return it.", "Returns Agent", "Sorry about that! Can you send a photo?"),
         ("sent it just now", "Returns Agent", "Got it; we'll collect it on Monday and refund you."),
         ("ok great", "Returns Agent", "Anything else?"),
         ("Will the delivery team take my old sofa away?", "Orders & Delivery Agent", None)]},
    {"domain": "energy supplier",
     "agents": {
         "Power Outages Agent": "Handles power cuts and faults: outages, flickering lights and engineer visits.",
         "Billing Agent": "Handles bills: high bills, charges and spreading payments.",
         "Moving Home Agent": "Moves the energy supply when a customer moves house."},
     "turns": [
         ("The power is out on my whole street.", "Power Outages Agent", "Sorry about that. What's your postcode?"),
         ("4412", "Power Outages Agent", "Engineers are on site; power should be back by 10 pm."),
         ("While I have you, why is my last bill so high?", "Billing Agent", "It includes two months of catch-up charges."),
         ("can I spread it out?", "Billing Agent", "Yes, over the next three bills."),
         ("We're moving house on the 14th.", "Moving Home Agent", "What's the new address?"),
         ("12 Harbour Road", "Moving Home Agent", "Done, your supply moves on the 14th."),
         ("The lights flickered again just now.", "Power Outages Agent", "We'll send an engineer. Morning or afternoon?"),
         ("morning please", "Power Outages Agent", None)]},
    {"domain": "online courses",
     "agents": {
         "Enrolment Agent": "Handles signing up for courses, start dates and course choice.",
         "Payments Agent": "Handles course payments: declined cards, payment links and invoices.",
         "Technical Help Agent": "Fixes problems with videos, logins and the learning platform."},
     "turns": [
         ("I want to sign up for the Python course.", "Enrolment Agent", "The next class starts on the 3rd. Shall I enrol you?"),
         ("yes", "Enrolment Agent", "Done. You'll get the details by email."),
         ("My card was declined when I tried to pay.", "Payments Agent", "Try another card, or I can send you a payment link."),
         ("send the link", "Payments Agent", "Sent."),
         ("The video lessons won't play on my laptop.", "Technical Help Agent", "Which browser are you using?"),
         ("Chrome", "Technical Help Agent", "Please clear the cache and try again."),
         ("that fixed it, thanks", "Technical Help Agent", "Great!"),
         ("Can I get an invoice for my company?", "Payments Agent", None)]},
]

if ROUTER_SMOKE:
    NEW_TESTS = NEW_TESTS[:2]                                                 # code check only

APPROACHES = ["router"]                           # Phase 7: the per-pass heads and "router x2" were removed


def route_probs(messages, agents, current):
    """Probabilities over the candidates from the final router."""
    return {"router": head_probs(final_heads, *candidate_features(messages, agents, current)).cpu()}


def short(name):
    return name.replace(" Agent", "")[:16]


turn_log = []                                     # every turn's probabilities, for later inspection
passes = {a: 0 for a in APPROACHES}
for conv in NEW_TESTS:
    names, messages, current = list(conv["agents"]), [], None
    print(f"\n{conv['domain']}  (candidates: {', '.join(short(n) for n in names)})")
    print(f"  {'turn':>4} | {'user message':40s} | {'current':16s} | {'expected':16s} | "
          + " | ".join(f"{a + ': pick':>23s}" for a in APPROACHES) + " | p(expected) " + " / ".join(APPROACHES))
    for t, (user_msg, expected, reply) in enumerate(conv["turns"], 1):
        messages.append({"role": "user", "content": user_msg})
        probs = route_probs(messages, conv["agents"], current)
        e = names.index(expected)
        picks = {a: names[int(torch.argmax(p_))] for a, p_ in probs.items()}
        for a in APPROACHES:
            passes[a] += picks[a] == expected
        turn_log.append({"domain": conv["domain"], "turn": t, "message": user_msg, "current": current, "expected": expected,
                         "probs": {a: dict(zip(names, probs[a].tolist())) for a in APPROACHES}})
        msg = user_msg if len(user_msg) <= 40 else user_msg[:37] + "..."
        cols = " | ".join(f"{short(picks[a]):>16s} {probs[a].max().item():.2f}{' x' if picks[a] != expected else '  '}" for a in APPROACHES)
        print(f"  {t:>4} | {msg:40s} | {short(current or 'none'):16s} | {short(expected):16s} | {cols} | "
              + " / ".join(f"{probs[a][e].item():.2f}" for a in APPROACHES))
        current = picks["router"]                 # live: the router's own pick answers, so it holds the next turn
        if reply:
            messages.append({"role": "assistant", "content": reply})
            print(f"  {'':>4} |   assistant ({short(current)}): {reply}")

n_turns = len(turn_log)
print(f"\nunseen-domain conversations, {n_turns} user turns:")
for a in APPROACHES:
    print(f"  {a:10s}: {passes[a]:>2} pass / {n_turns - passes[a]:>2} fail")

---
# Phase 16 — A simpler question first: has the context changed?

## Cell 60 — Context-change labels (Y / N)

**What.** Before choosing among agents, ask one yes/no question about the **last user message**: does it move the conversation to a **different topic** (Y), or does it continue the current one (N)? The generic data now labels every conversation that has a previous turn:
- **N:** follow-ups ("can you check again?"), short answers to the agent's question ("Sunday", "PS5", "12 Harbour Road"), closings ("thanks", "got it"), and "ok" after a handoff. The user is still on the same thread.
- **Y:** topic changes ("Also, my bill is double"), short switches ("and the bill?", "printer please"), returning to an earlier topic, and "forget the X, I want Y".

**Why.** Most routing failures so far were short replies that the router sent back to the conversation's *first* topic. Telling "same thread" from "new topic" is a simpler, candidate-free question. If it can be answered (close to) perfectly, it can later steer the router: on N keep the current agent, on Y let the compatibility head choose.

**More short replies.** This data adds many more short samples: 4 short answers per specialist, 2 closings per specialist, 16 short answers after a topic change and 16 short switches per domain. Short Y examples matter, or the detector would just learn "short means N".

**Expect.** The Y/N counts, how many last messages are short, and one example of each kind.

**Data fixes for gaps found in run 5:**
- "Also …" / "And …" follow-ups that stay on the same topic (N);
- returns to an earlier topic with "… again" (Y);
- the first real request after the General agent's prompt (Y);
- more short-switch wording ("password?", "ok and billing").

(Phase 1 also trained a second label, **agent switch**, Y when the right agent differs from the current one. It never beat context change as a gate and was removed in Phase 7.)


In [ ]:
# Cell 60 — Context-change labels: does the LAST user message move the conversation to a different topic? (Y / N)

cc_conv_idx = [i for i, ex in enumerate(generic_conversations) if ex["context_change"] is not None]   # needs a previous turn
cc_examples = [generic_conversations[i] for i in cc_conv_idx]
cc_labels = torch.tensor([float(ex["context_change"]) for ex in cc_examples])
cc_fold = torch.tensor([next(f for f, doms in enumerate(FOLDS["route"]) if ex["domain"] in doms) for ex in cc_examples])


def is_short(ex):
    return len(ex["messages"][-1]["content"].split()) <= 3


n_yes = int(cc_labels.sum())
short_ex = [ex for ex in cc_examples if is_short(ex)]
print(f"{len(cc_examples)} generic conversations with a previous turn: {n_yes} Y (context changes) | {len(cc_examples) - n_yes} N (same context)")
print(f"short last messages (<= 3 words): {len(short_ex)} | Y {sum(ex['context_change'] for ex in short_ex)} | "
      f"N {sum(not ex['context_change'] for ex in short_ex)}")

for tag in ["short_answer", "closing", "short_switch", "topic_change", "ok_after_handoff", "stay_prefix", "return_again", "after_general"]:
    ex = next((e for e in cc_examples if tag in e["tags"] and (tag != "short_answer" or "closing" not in e["tags"])), None)
    if ex is None:
        continue
    print(f"\n[{tag}]  current agent: {ex['current_agent']}  ->  context change: {'Y' if ex['context_change'] else 'N'}")
    for m in ex["messages"][-3:]:
        print(f"   {m['role']:9s}: {m['content']}")

## Cell 61 — Detector features from one joint pass

**What.** One ModernBERT pass over the conversation plus its current agent (`history [SEP] current agent: X`), the same text as the dual pass's conversation input. From that pass, three spans are mean-pooled:
- the **last user message**: its tokens have attended to the whole history;
- **everything before it**: the context so far;
- the **assistant message just before it**: often a question ("Which platform?") that a short answer refers to.

**Features.** Standardize, then the three vectors plus how the last message compares with the history and with the previous assistant message (products and absolute differences): 7 × H numbers per conversation. Nothing is reduced.

**Expect.** About 4,300 passes on the first run (a few minutes on the T4), cached afterwards.

In [ ]:
# Cell 61 — Features for the detector: one joint pass over (history [SEP] current agent), three spans pooled

CC_CACHE = CACHE_DIR / f"context_features_{DETECTOR_ENC.tag}.pt"     # the DETECTOR encoder
cc_cache = torch.load(CC_CACHE) if CC_CACHE.exists() else {}


def context_features(examples):
    """-> [N, 3, H]: mean of the last user message, of everything before it, and of the assistant message just before it.

    All three come from ONE pass over the whole conversation, so the last message's tokens have read the history."""
    built = [build_text(ex["messages"], ex["current_agent"], None, DETECTOR_ENC.query, enc=DETECTOR_ENC) for ex in examples]
    ranges = [[sp["messages"][-1], (sp["messages"][0][0], sp["messages"][-2][1]), sp["messages"][-2]] for _, sp in built]
    return span_means([t for t, _ in built], ranges, cc_cache, CC_CACHE, "context features", mix=False, enc=DETECTOR_ENC)[:, 0]


def context_matching(feats, stats):
    """Standardize; then last message, history, previous assistant message, and how the last message compares to both -> [N, 7H]."""
    z = (feats - stats[0]) / stats[1]
    l, h, a = z[:, 0], z[:, 1], z[:, 2]
    return torch.cat([l, h, a, l * h, (l - h).abs(), l * a, (l - a).abs()], dim=1)


cc_feats = context_features(cc_examples)
print(f"context features {tuple(cc_feats.shape)} (conversations, [last message, history, previous assistant message], H)")

## Cell 62 — Train and score the detector on unseen domains

**What.** The same leave-domains-out scheme as the router: for each fold, train on 3 folds, early-stop on the next one, and score the held-out fold. The head is MLP-256 (fixed in Phase 7; it won in run 11), trained with a class-balanced loss, so Y and N count equally.

**Expect.** Held-out results in absolute numbers. "Y found" counts real topic changes that were detected, and "N kept" counts same-thread messages left alone. Both are reported for all messages and for short ones (≤ 3 words), because short replies are the hard case.

In [ ]:
# Cell 62 — Train the Y/N detector with leave-domains-out folds; score the held-out domains

CC_HEADS = ["mlp-256"]                     # Phase 7: fixed (it won in run 11; linear / mlp-64 remain available in make_head)


def train_binary(kind, X_tr, y_tr, X_va=None, y_va=None, epochs=MAX_EPOCHS, patience=PATIENCE, seed=42):
    """X: [N, F], y: [N] in {0, 1}; class-balanced BCE. Early stopping on (X_va, y_va) if given."""
    torch.manual_seed(seed)
    head = make_head(kind, X_tr.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    pos_weight = (1 - y_tr).sum() / y_tr.sum().clamp(min=1)               # Y and N count equally
    loss_fn = lambda X, y: F.binary_cross_entropy_with_logits(head(X).squeeze(-1), y, pos_weight=pos_weight)
    g = torch.Generator().manual_seed(seed)
    best, history = (float("inf"), None, 0), []
    for epoch in range(1, epochs + 1):
        head.train()
        for idx in torch.randperm(len(X_tr), generator=g).split(64):
            idx = idx.to(DEVICE)
            loss = loss_fn(X_tr[idx], y_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        if X_va is None:
            continue
        head.eval()
        with torch.no_grad():
            history.append((loss_fn(X_tr, y_tr).item(), loss_fn(X_va, y_va).item()))
        if history[-1][1] < best[0]:
            best = (history[-1][1], copy.deepcopy(head.state_dict()), epoch)
        if epoch - best[2] >= patience:
            break
    if X_va is not None:
        head.load_state_dict(best[1])
    head.eval()
    return head, best[2], history


def confusion(pred, y):
    """-> dict of Y/N counts: tp (said Y, was Y), fn (said N, was Y), tn, fp."""
    pred, y = pred.bool(), y.bool()
    return {"tp": int((pred & y).sum()), "fn": int((~pred & y).sum()), "tn": int((~pred & ~y).sum()), "fp": int((pred & ~y).sum())}


def fmt_cc(c):
    yes, no = c["tp"] + c["fn"], c["tn"] + c["fp"]
    return (f"{c['tp'] + c['tn']:>5} / {yes + no:<5} correct | Y: {c['tp']:>4}/{yes:<4} found | "
            f"N: {c['tn']:>4}/{no:<4} kept")


short_mask = torch.tensor([is_short(ex) for ex in cc_examples])


def lodo_detector(labels, title):
    """Leave-domains-out training and scoring of every head type for one Y/N label -> (results, best head type)."""
    t0 = time.perf_counter()
    results = {k: {"all": Counter(), "short": Counter(), "best_epochs": []} for k in CC_HEADS}
    log(f"{title} detector: {len(CC_HEADS)} heads x {N_FOLDS} folds")
    for f in range(N_FOLDS):
        te, va = cc_fold == f, cc_fold == (f + 1) % N_FOLDS
        tr = ~(te | va)
        stats = (cc_feats[tr].mean(0, keepdim=True), cc_feats[tr].std(0, keepdim=True) + 1e-6)
        X, y = context_matching(cc_feats, stats).to(DEVICE), labels.to(DEVICE)
        for kind in CC_HEADS:
            head, best_epoch, _ = train_binary(kind, X[tr.to(DEVICE)], y[tr.to(DEVICE)], X[va.to(DEVICE)], y[va.to(DEVICE)])
            with torch.no_grad():
                pred = (head(X[te.to(DEVICE)]).squeeze(-1) > 0).cpu()
            results[kind]["all"].update(confusion(pred, labels[te]))
            results[kind]["short"].update(confusion(pred[short_mask[te]], labels[te][short_mask[te]]))
            results[kind]["best_epochs"].append(best_epoch)
            log(f"  fold {f} | {kind:8s} | best epoch {best_epoch:3d} | held-out: {fmt_cc(confusion(pred, labels[te]))}")
        del X, y
    print(f"\n{title}: held-out domains (every domain held out once), {time.perf_counter() - t0:.0f}s")
    for kind in CC_HEADS:
        print(f"  {kind:8s} all messages   : {fmt_cc(results[kind]['all'])}")
        print(f"  {'':8s} short (<= 3 w) : {fmt_cc(results[kind]['short'])}")
    best = max(CC_HEADS, key=lambda k: results[k]["all"]["tp"] + results[k]["all"]["tn"])
    print(f"best {title} head: {best} | best epochs per fold {results[best]['best_epochs']}\n")
    return results, best


cc_results, CC_BEST = lodo_detector(cc_labels, "context-change")


## Cell 63 — Final detector, tested on realistic conversations

**What.** Train the best detector head on all generic data. Then test it on conversations it has never seen:
- **(a)** the realistic conversations of Cell 59. A turn is Y when its expected agent differs from the previous turn's. The current agent is the previous turn's **correct** agent, so this measures the detector alone, without the router's mistakes.
- **(b)** the banking ding-dong turns: `topic_change` and `ding_dong` turns are Y, `same_topic` turns are N.

**Expect.** Every turn with its truth, `p(change)` and the detector's answer, then a summary. The goal is 100%. If it gets there, the next step is to combine it with the router: on N stay with the current agent, on Y let the compatibility head choose.

In [ ]:
# Cell 63 — Final detector on all generic data; test it on the realistic conversations (Cell 59) and banking ding-dong turns

CC_STATS = (cc_feats.mean(0, keepdim=True), cc_feats.std(0, keepdim=True) + 1e-6)
cc_epochs = max(1, int(statistics.median(cc_results[CC_BEST]["best_epochs"])))
cc_head, _, _ = train_binary(CC_BEST, context_matching(cc_feats, CC_STATS).to(DEVICE), cc_labels.to(DEVICE), epochs=cc_epochs)
log(f"final context-change detector: {CC_BEST}, {cc_epochs} epochs, trained on {len(cc_examples)} generic conversations")


def p_change(messages, current):
    """Probability that the last user message changes the context, given the history and the agent holding it."""
    feats = context_features([{"messages": messages, "current_agent": current}])
    with torch.no_grad():
        return torch.sigmoid(cc_head(context_matching(feats, CC_STATS).to(DEVICE))).item()


def show(rows, title):
    print(f"\n{title}")
    print(f"  {'user message':44s} | {'current':18s} | truth | p(change) | said | ")
    for msg, cur, truth, p in rows:
        said = p > 0.5
        m_ = msg if len(msg) <= 44 else msg[:41] + "..."
        print(f"  {m_:44s} | {(cur or 'none').replace(' Agent', '')[:18]:18s} |   {'Y' if truth else 'N'}   |   {p:5.2f}   |  {'Y' if said else 'N'}  | {'' if said == truth else 'x'}")
    c = confusion(torch.tensor([p > 0.5 for *_, p in rows]), torch.tensor([t for _, _, t, _ in rows]))
    print(f"  -> {fmt_cc(c)}")
    return c


# (a) the realistic conversations of Cell 59: label = the expected agent differs from the previous turn's; current = previous expected agent
realistic = []
for conv in NEW_TESTS:
    messages, prev = [], None
    for user_msg, expected, reply in conv["turns"]:
        messages.append({"role": "user", "content": user_msg})
        if prev is not None:
            realistic.append((user_msg, prev, expected != prev, p_change(list(messages), prev)))
        prev = expected
        if reply:
            messages.append({"role": "assistant", "content": reply})
c_real = show(realistic, f"realistic conversations (Cell 59), {len(realistic)} turns after the first:")

# (b) banking ding-dong: topic_change / ding_dong turns are Y, same_topic turns are N (start and ambiguous turns skipped)
banking_turns = []
for seq in router_dingdong:
    prev = None
    for u in seq["user_turns"]:
        if prev is not None and u["turn_type"] in ("topic_change", "ding_dong", "same_topic"):
            prefix = seq["messages"][:u["message_index"] + 1]
            banking_turns.append((prefix[-1]["content"], prev, u["turn_type"] != "same_topic", p_change(prefix, prev)))
        prev = u["expected_agent"]
c_bank = show(banking_turns, f"banking ding-dong turns, {len(banking_turns)} turns:")

held = cc_results[CC_BEST]["all"]
print(f"\nSUMMARY (Y/N: does the last user message change the context?)")
print(f"  held-out generic domains : {fmt_cc(held)}")
print(f"  realistic conversations  : {fmt_cc(c_real)}")
print(f"  banking ding-dong turns  : {fmt_cc(c_bank)}")


## Cell 69 — E10: the realistic test set

**What.** 35 free-form conversations from 13 companies that appear nowhere in training: a pharmacy, a water utility, a dating app, a ride-hailing app, a cruise line, a solar installer, a pet-food subscription, a photo-printing service, a car dealership, a driving school, a parking operator, an electronics repair shop and a games-console company. Each company has 3 broad agents, and there are **193 user turns** in total. The conversations were written without templates (`data/modernbert_router/build_realistic_test_data.py`) and are never used for training.

**Live routing.** Every user turn is routed with the history so far. Each approach's **own previous pick** is its current agent, so mistakes can carry over, as in deployment. This cell scores the **router alone**; Cell 70 adds the gate. (The cross-only head, hard cascade and soft fusion were compared here until Phase 7.)

**Turn types.** Every turn is labelled, so the table shows *where* each approach fails:

| Type | Meaning |
|---|---|
| start | first message |
| answer | a short answer to the agent's question |
| follow | a follow-up on the same request |
| aspect | the same agent, a different aspect |
| closing | "ok cheers" |
| also_stay | "Also …" that stays with the agent |
| switch | a clear switch |
| short_switch | a switch in a few words |
| return | back to an earlier topic |
| handoff_ok | "ok" after a handoff |
| branch | the same words, a different agent depending on the history |

**Expect.** Pass / total per turn type and in total, then the router's first mistakes.

In [ ]:
# Cell 69 — E10: the realistic test set (35 conversations, 13 companies, 193 typed user turns), routed live

REAL_FILE = DATA_DIR / "realistic" / "conversations.json"
if not REAL_FILE.exists():                                     # a data clone made before this file existed: refresh it
    repo_dir = DATA_DIR.parent.parent
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
real_convs = load(REAL_FILE)
if globals().get("ROUTER_SMOKE"):
    real_convs = real_convs[:3]                                # code check only


def pick_router(messages, agents, current):
    return list(agents)[int(torch.argmax(score_candidates(messages, agents, current)))]


E10_APPROACHES = {"router": pick_router}         # Phase 7: cross-only, hard cascade and soft fusion were removed; the gate is Cell 70

log(f"E10: {len(real_convs)} conversations, {sum(len(c['user_turns']) for c in real_convs)} user turns x {len(E10_APPROACHES)} approaches")
e10_rows = []
for k, conv in enumerate(real_convs):
    current = {a: None for a in E10_APPROACHES}                # live: each approach's own previous pick holds the next turn
    for u in conv["user_turns"]:
        prefix = conv["messages"][:u["message_index"] + 1]
        row = {"id": conv["id"], "company": conv["company"], "type": u["turn_type"], "expected": u["expected_agent"],
               "message": prefix[-1]["content"]}
        for a, pick_fn in E10_APPROACHES.items():
            row[a] = current[a] = pick_fn(prefix, conv["agents"], current[a])
        e10_rows.append(row)
    if (k + 1) % 10 == 0:
        log(f"  E10 {k + 1}/{len(real_convs)} conversations")

TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]
print(f"\nE10 realistic test, live | router encoder: {ROUTER_ENCODER} | detector encoder: {DETECTOR_ENCODER} | router: {BEST_CFG}")
print(f"{'turn type':14s} | {'turns':>5} | " + " | ".join(f"{a:>12s}" for a in E10_APPROACHES))
print("-" * (24 + 15 * len(E10_APPROACHES)))
for tt in TURN_TYPES + ["ALL"]:
    rows = [r for r in e10_rows if tt == "ALL" or r["type"] == tt]
    if not rows:
        continue
    if tt == "ALL":
        print("-" * (24 + 15 * len(E10_APPROACHES)))
    print(f"{tt:14s} | {len(rows):>5} | " + " | ".join(f"{sum(r[a] == r['expected'] for r in rows):>6} / {len(rows):<3}"
                                                     for a in E10_APPROACHES))

print("\nrouter mistakes (first 15):")
for r in [r for r in e10_rows if r["router"] != r["expected"]][:15]:
    msg = r["message"] if len(r["message"]) <= 50 else r["message"][:47] + "..."
    print(f"  {r['id']:9s} {r['type']:12s} | {msg:50s} | expected {r['expected'].replace(' Agent', ''):22s} | got {r['router'].replace(' Agent', '')}")

## Cell 70 — A two-sided gate: "confident stay" and "confident switch"

**Why.** The context-change detector is very reliable at its extremes, and the router has one remaining weakness on real conversations: when a user moves to a **closely related agent** with a message that still mentions the current topic ("and is the snorkelling trip still available?" while talking to Onboard Services), the router stays with the current agent. Run 13 traced every failed E10 test switch to this.

**The gate.** Use the detector only where it is very sure, on either side:

```text
p = detector(conversation, current agent)          # p(context change): ModernBERT pooled (Cells 61-63) or the unified model's yes / no answer
p < τ_low    (sure nothing changed)  → keep the current agent                 "confident stay"  (runs 10-13)
p > τ_high   (sure the topic changed) → the router picks among the OTHER agents "confident switch" (new)
otherwise                            → the router picks among ALL agents
```

This is not the Phase 1 hard cascade: that acted on every p ≥ 0.5, with the weaker agent-switch detector, and so excluded the right agent whenever the detector was unsure. Here the exclusion happens only at the very top of the detector's range.

**Choosing τ honestly.** E10's 13 companies are split in two:
- **7 dev companies:** τ_low ∈ {0.02, 0.05, 0.1, 0.2, 0.3} first; then, with that τ_low, τ_high ∈ {off, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999}. Ties go to "off" (the simpler gate), then to the higher τ_high (the more cautious one);
- **6 test companies:** the results are reported there, next to the router alone and the stay-only gate.

**Expect.** The dev sweeps, the test table per turn type, and every test turn whose answer the confident switch changed (fixed or broken). If it works, test switches should rise without losing follow-ups and "aspect" turns. (Q-K-V detectors were removed after run 13: no gain in either mode.)


In [ ]:
# Cell 70 — Two-sided gate on E10 (ModernBERT pooled context-change detector + route head); taus on dev companies

companies = sorted({c["company"] for c in real_convs})
DEV_COMPANIES = set(companies[::2])                                   # 7 companies to choose the taus on
TEST_COMPANIES = set(companies) - DEV_COMPANIES                       # 6 companies to report on
dev_convs = [c for c in real_convs if c["company"] in DEV_COMPANIES]
test_convs = [c for c in real_convs if c["company"] in TEST_COMPANIES]

TAUS_LOW = [0.02, 0.05, 0.1, 0.2, 0.3]
TAUS_HIGH = [None, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999]              # None = no confident switch
_memo = {}


def memo(key, fn):
    """The same (turn, current agent) always gives the same answer: compute it once across all taus."""
    if key not in _memo:
        _memo[key] = fn()
    return _memo[key]


def pick_other(messages, agents, current):
    """The router's best agent with the current one excluded."""
    names = list(agents)
    s = score_candidates(messages, agents, current).clone()
    s[names.index(current)] = float("-inf")
    return names[int(torch.argmax(s))]


DETECTORS = {"pooled": p_change}                    # run 17: routing data only (the context-change question is not asked to a head)
DET = "pooled"                                      # set per sweep below


def run_live(convs, tau_low=None, tau_high=None):
    """Route every turn live (the gate's own previous pick is the current agent).
    p < tau_low -> keep the current agent; p > tau_high -> router among the OTHER agents; else router among ALL agents."""
    rows = []
    for conv in convs:
        cur = None
        for k, u in enumerate(conv["user_turns"]):
            prefix = conv["messages"][:u["message_index"] + 1]
            action, p = "route", None
            if (tau_low is not None or tau_high is not None) and cur is not None and cur in conv["agents"]:
                p = memo(("p", DET, conv["id"], k, cur), lambda: DETECTORS[DET](prefix, cur))
                if tau_low is not None and p < tau_low:
                    action = "stay"
                elif tau_high is not None and p > tau_high:
                    action = "switch"
            if action == "route":
                cur = memo(("r", conv["id"], k, cur), lambda: pick_router(prefix, conv["agents"], cur))
            elif action == "switch":
                cur = memo(("o", conv["id"], k, cur), lambda: pick_other(prefix, conv["agents"], cur))
            rows.append({"id": conv["id"], "type": u["turn_type"], "message": prefix[-1]["content"], "expected": u["expected_agent"],
                         "pick": cur, "p": p, "action": action, "ok": cur == u["expected_agent"]})
    return rows


n_ok = lambda rows: sum(r["ok"] for r in rows)
dev_turns = sum(len(c["user_turns"]) for c in dev_convs)
test_turns = sum(len(c["user_turns"]) for c in test_convs)
log(f"gate: dev {len(DEV_COMPANIES)} companies / {dev_turns} turns, test {len(TEST_COMPANIES)} companies / {test_turns} turns")

# 1) DEV, per detector: tau_low for confident stay, then tau_high for confident switch (with that tau_low)
print(f"DEV ({', '.join(sorted(DEV_COMPANIES))})")
print(f"  router alone: {n_ok(run_live(dev_convs))} / {dev_turns}")
chosen = {}
for DET in DETECTORS:
    low_scores = [n_ok(run_live(dev_convs, t)) for t in TAUS_LOW]
    t_low = TAUS_LOW[max(range(len(TAUS_LOW)), key=lambda i: (low_scores[i], -i))]
    high_scores = [n_ok(run_live(dev_convs, t_low, t)) for t in TAUS_HIGH]
    order = sorted(range(len(TAUS_HIGH)), key=lambda i: (-high_scores[i], TAUS_HIGH[i] is not None, -(TAUS_HIGH[i] or 0)))
    chosen[DET] = (t_low, TAUS_HIGH[order[0]])
    print(f"  [{DET}] confident stay   | " + " | ".join(f"{t}: {s_}" for t, s_ in zip(TAUS_LOW, low_scores)) + f"  -> tau_low {t_low}")
    print(f"  [{DET}] + confident switch | " + " | ".join(f"{t or 'off'}: {s_}" for t, s_ in zip(TAUS_HIGH, high_scores))
          + f"  -> tau_high {chosen[DET][1] or 'off'}")
log("gate dev sweeps done")

# 2) TEST with the dev-chosen taus, per detector
test_rows = {"router alone": run_live(test_convs)}
for DET in DETECTORS:
    t_low, t_high = chosen[DET]
    test_rows[f"{DET} stay @{t_low}"] = run_live(test_convs, t_low)
    test_rows[f"{DET} stay+switch @{t_high or 'off'}"] = run_live(test_convs, t_low, t_high)
print(f"\nTEST ({', '.join(sorted(TEST_COMPANIES))}) — taus chosen on DEV only")
names = list(test_rows)
print(f"{'turn type':14s} | {'turns':>5} | " + " | ".join(f"{n:>24s}" for n in names))
for tt in TURN_TYPES + ["ALL"]:
    sel = {n: [r for r in rows if tt == "ALL" or r["type"] == tt] for n, rows in test_rows.items()}
    if not sel["router alone"]:
        continue
    total = len(sel["router alone"])
    print(f"{tt:14s} | {total:>5} | " + " | ".join(f"{n_ok(sel[n]):>17} / {total:<4}" for n in names))

# Which test turns did each detector's confident switch change?
for DET in DETECTORS:
    t_low, t_high = chosen[DET]
    stay_rows, both_rows = test_rows[f"{DET} stay @{t_low}"], test_rows[f"{DET} stay+switch @{t_high or 'off'}"]
    changed = [(a_, b_) for a_, b_ in zip(stay_rows, both_rows) if a_["pick"] != b_["pick"]]
    print(f"\n[{DET}] test turns changed by the confident switch: {len(changed)} "
          f"(fixed {sum(b_['ok'] and not a_['ok'] for a_, b_ in changed)}, broken {sum(a_['ok'] and not b_['ok'] for a_, b_ in changed)})")
    for a_, b_ in changed:
        m_ = b_["message"] if len(b_["message"]) <= 50 else b_["message"][:47] + "..."
        print(f"  {b_['id']:9s} {b_['type']:12s} | {m_:50s} | expected {b_['expected'].replace(' Agent', ''):20s} | "
              f"{a_['pick'].replace(' Agent', ''):20s} -> {b_['pick'].replace(' Agent', '')}")


## Cell 71 — E11: first-turn routing only

**What.** The core skill on its own, with no conversation effects: **one incoming message, no history, no current agent**. Pick the right agent among **5 candidates** per company.

- **100 messages** from 10 companies that appear nowhere else: online grocery, a smartwatch maker, council waste services, an online marketplace for sellers, a home-appliance maker, a cinema chain, a bike-share scheme, accounting software, a hair-salon chain and a charity.
- Each company has 5 agents, including deliberately **close pairs** (for example Bin Collections vs Garden Waste, Disputes vs Shipping Labels).
- The messages are written free-form (`data/modernbert_router/build_first_turn_test_data.py`) and are never used for training.

**Message types.**

| Type | Meaning | Example |
|---|---|---|
| clear | names the need | "I need a new water filter for my fridge." |
| short | 1–4 words | "flat tyre", "payout delayed" |
| indirect | describes the situation without the keyword | "I'm retired and would love to help out a couple of days a week." |
| close | between two similar agents | "my garden waste bin was skipped this week" |
| noisy | typos / lower case | "can i add bananas to tomorrows order" |

**Expect.** Router top-1 and top-2 accuracy per type, per company, and a list of every router mistake with its confidence. Top-2 shows how often the right agent is at least the runner-up, which matters for a "shortlist and confirm" design.

In [ ]:
# Cell 71 — E11: first-turn routing only (one incoming message, no history, no current agent), 5 agents per company

FT_FILE = DATA_DIR / "first_turn" / "messages.json"
if not FT_FILE.exists():                                       # a data clone made before this file existed: refresh it
    repo_dir = DATA_DIR.parent.parent
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
ft = load(FT_FILE)
ft_companies, ft_msgs = ft["companies"], ft["messages"]
log(f"E11: {len(ft_msgs)} first messages, {len(ft_companies)} companies x 5 agents")

ft_rows = []
for m in ft_msgs:
    agents = ft_companies[m["company"]]
    names = list(agents)
    feats = candidate_features([{"role": "user", "content": m["message"]}], agents, None)
    p_router = head_probs(final_heads, *feats).cpu()
    ft_rows.append({**m, "router": names[int(p_router.argmax())],
                    "router_top2": [names[i] for i in p_router.topk(2).indices.tolist()], "p_router": p_router.max().item()})

FT_TYPES = ["clear", "short", "indirect", "close", "noisy"]
print(f"E11 first-turn routing | router encoder: {ROUTER_ENCODER} | router: {BEST_CFG}")
print(f"{'type':10s} | {'msgs':>4} | {'router':>9} | {'router top-2':>12}")
for tt in FT_TYPES + ["ALL"]:
    rows = [r for r in ft_rows if tt == "ALL" or r["type"] == tt]
    top1 = sum(r["router"] == r["target_agent"] for r in rows)
    top2 = sum(r["target_agent"] in r["router_top2"] for r in rows)
    print(f"{tt:10s} | {len(rows):>4} | {top1:>4} / {len(rows):<3} | {top2:>6} / {len(rows):<3}")

print("\nper company (router):")
for comp in ft_companies:
    rows = [r for r in ft_rows if r["company"] == comp]
    print(f"  {comp:32s} {sum(r['router'] == r['target_agent'] for r in rows):>2} / {len(rows)}")

print("\nrouter mistakes:")
for r in [r for r in ft_rows if r["router"] != r["target_agent"]]:
    msg = r["message"] if len(r["message"]) <= 58 else r["message"][:55] + "..."
    print(f"  {r['type']:8s} | {msg:58s} | expected {r['target_agent'].replace(' Agent', ''):26s} | got {r['router'].replace(' Agent', '')} ({r['p_router']:.2f})")

## Cell 72 — E12: grounded choice questions

**What.** Can the router, exactly as trained for routing, answer **general choice questions** about a text? This is the zero-shot starting point of phase 8 and its comparison with the Strands decider, a model built for exactly this.

**The test (E12, `data/modernbert_router/choice_test/items.json`).** 170 items written for this test and never used in training. Every answer can be read from the text itself; none needs world knowledge. There are three question types, the decider's three primitives:

| Type | Items | Example |
|---|---|---|
| yes / no | 58 | "Help! My payouts have been failing for 3 days!" — *Does this convey urgency?* → yes |
| choice | 59 | same text — *Which team should handle this?* billing / sales / retail → billing |
| score | 53 | same text — *How frustrated is the writer?* calm / frustrated / furious → frustrated |

Eleven texts are asked several different questions (25 items), which checks that a model reads the **question**, not only the text.

**Asked to the choice head (run 17).** The text becomes the user's message, the item's question goes in the `question:` segment, and each option is a candidate. Yes / no and choice take the top option. Score reports the top level and the expected level Σ p × index. The choice head was trained only on general questions of the same three types (Cell 41), on different texts. E12 itself is never used in training. Run 14 asked the same items zero-shot to the routing-only model: **96 / 170**; run 16's single shared head: **113 / 170**. The decider scored **160 / 170**.

**Expect.** Pass / total per type and per category, the score errors in levels, the question-group check, the mistakes, and the time per question. The comparison with run 14 shows what the choice training added.


In [ ]:
# Cell 72 — E12: grounded choice questions, asked to the CHOICE head (E12 items are never used in training)

CT_FILE = DATA_DIR / "choice_test" / "items.json"
if not CT_FILE.exists():                                       # a data clone made before this file existed: refresh it
    repo_dir = DATA_DIR.parent.parent
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
e12_data = load(CT_FILE)
e12_items = e12_data["items"][:12] if ROUTER_SMOKE else e12_data["items"]


def choice_probs(item):
    """The text as the user's message, the item's question, its options -> probabilities in option order."""
    return torch.tensor(list(ask([{"role": "user", "content": item["state"]}], item["question"], item["options"]).values()))


choice_probs(e12_items[0])                                     # warm-up (not timed)
e12_rows, t0 = [], time.perf_counter()
for it in e12_items:
    p = choice_probs(it)
    top = int(p.argmax())
    row = {**it, "probs": p.tolist(), "pick": it["options"][top], "ok": it["options"][top] == it["answer"]}
    if it["type"] == "score":
        row["expected_level"] = float(sum(i * pi for i, pi in enumerate(p.tolist())))
        row["level_error"] = abs(top - it["answer_index"])
    e12_rows.append(row)
e12_ms = (time.perf_counter() - t0) * 1000 / len(e12_items)
log(f"E12: {len(e12_rows)} items, {e12_ms:.0f} ms per question")

print(f"E12 grounded choice | choice head | encoder: {ROUTER_ENCODER} | {BEST_CFG} | trained on {len(choice_train)} general questions only")
print(f"{'type':8s} | {'items':>5} | {'correct':>9}")
for t in ["noul", "choice", "score", "ALL"]:
    rows = [r for r in e12_rows if t == "ALL" or r["type"] == t]
    print(f"{t:8s} | {len(rows):>5} | {sum(r['ok'] for r in rows):>4} / {len(rows):<4}")
sc = [r for r in e12_rows if r["type"] == "score"]
if sc:
    print(f"score: exact level {sum(r['level_error'] == 0 for r in sc)} / {len(sc)} | within 1 level {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)} | "
          f"mean |expected - true| {sum(abs(r['expected_level'] - r['answer_index']) for r in sc) / len(sc):.2f} levels")
yes_rows = [r for r in e12_rows if r["type"] == "noul"]
print(f"yes / no: answered yes {sum(r['pick'] == 'yes' for r in yes_rows)} of {len(yes_rows)} (truth: {sum(r['answer'] == 'yes' for r in yes_rows)} yes)")

print("\nper category (all types):")
for cat in sorted({r["category"] for r in e12_rows}):
    rows = [r for r in e12_rows if r["category"] == cat]
    print(f"  {cat:10s} {sum(r['ok'] for r in rows):>3} / {len(rows)}")

groups = {}
for r in e12_rows:
    if "pair" in r:
        groups.setdefault(r["pair"], []).append(r)
print(f"\nquestion groups (same text, different questions): all answers right in {sum(all(r['ok'] for r in g) for g in groups.values())} of {len(groups)} groups")
for pid, g in groups.items():
    print(f"  {pid}: " + " | ".join(f"{r['question'][:34]} -> {r['pick']}{'' if r['ok'] else ' x'}" for r in g))

print("\nmistakes:")
for r in [r for r in e12_rows if not r["ok"]]:
    s_ = r["state"] if len(r["state"]) <= 46 else r["state"][:43] + "..."
    print(f"  {r['id']} {r['type']:6s} | {s_:46s} | {r['question'][:38]:38s} | expected {r['answer']:14s} | got {r['pick']} ({max(r['probs']):.2f})")

with open("e12_ours.json", "w") as fh:                         # for the scorecard
    json.dump({"ms_per_question": e12_ms, "rows": [{k: r[k] for k in ("id", "type", "category", "pick", "ok", "probs")} for r in e12_rows]}, fh)


---
# Phase 17 — What we tried and dropped

Over 17 runs this notebook grew one idea at a time, and most ideas lost. The rule we used to decide, from Phase 7 on: tune every option on the 7 **dev** companies of the realistic test set (E10) and judge it on the 6 **test** companies (83 turns) it never saw. A component had to beat the simpler alternative by **at least 2 of those 83 turns** to stay; on a tie, the simpler or cheaper option won. Every number below is correct / total, from the run named in brackets. The full entries are in `experiments/modernbert_router_ledger.md`. The code of each dropped idea is kept in git history at the commit of its run.

### Router design

| Idea | What it did | Result | Why it was dropped |
|---|---|---|---|
| **PCA on the features** (run 3) | Reduce the features to 256 or 64 dimensions before the head; an 18-config grid per run | The winner used PCA 256; without PCA the heads overfit in 1–5 epochs | Removed after run 3 (a project decision, to keep the features whole and the grid small); not re-tested |
| **"Router x2"** (run 4) | Repeat the last user message in the input | E5: 29 vs 35 / 40 for the plain router | Hurt |
| **Dual-only and cross-only heads** (runs 1–11) | Score from the standalone embeddings only, or from the joint pass only | Held-out (run 11): dual 4,349, cross 5,280, combined **5,722** of 6,658. Cross-only was +4 on E10 in run 10 but −4 in run 11 | The combined head (both passes + similarities) was better or equal wherever results were stable |
| **Layer mix** (run 8) | Mean of the last 4 layers instead of the last layer | Worse than "last" for every feature set and head, by 1–2 points | Lost everywhere |
| **Per-run head selection** (runs 1–11) | Each run chose among MLP-64, MLP-256 and a residual head | Differences < 1% held-out, but in run 11 the selection switched heads at the same time as the data changed, so we couldn't tell which caused the result | Heads are fixed since Phase 7: router MLP-256, detector MLP-256 |
| **Choosing on generic held-out data** (run 8) | Pick models and weights by leave-domains-out accuracy on the template data | The winning head scored +57 there but went down on banking and realistic conversations; soft fusion added +423 there but lost 13 on banking | The template data rewards cues that real conversations don't have. That's why dev / test companies from the realistic set decide from run 9 on |

### Combining the router with a yes / no detector

| Idea | What it did | Result | Why it was dropped |
|---|---|---|---|
| **Agent-switch detector** (run 6) | A second yes / no model: "should a different agent answer?" | As the gate's detector: 73 vs **74** / 83 for the context-change detector (run 10); kept only 5 of 27 "stay" cases on banking | Never better than the context-change detector, doubled the training, and partly repeated the router's job |
| **Hard cascade** (run 6) | The detector decides stay / switch; on "switch" the current agent is excluded | Below the router alone everywhere: realistic 30 vs **34** / 40, banking 143 vs **176** / 225; E10 117 vs **154** / 193 (run 10) | A wrong "switch" became a guaranteed routing error, and errors carried forward live ("ok thanks" then stayed with the wrong agent) |
| **Soft fusion** (runs 8–11) | Router probabilities + w × detector evidence | Run 9: realistic +3, E10 +1, banking −10 | Mixed across test sets; the confident-stay gate does the useful part more simply |
| **Q-K-V attention head** (runs 7, 13) | A learned attention layer: the current message's tokens (Q) attend to the history (K, V) | Run 7 looked better out of domain, but run 8 didn't confirm it. Run 13 rewrite, current message encoded jointly or alone: 70 / 70 / 70 in the gate, held-out 5,677 vs **5,728** for pooled | The attention weights stayed almost uniform (≈ 1 / number of history tokens, on function words): pooling with extra parameters, and 17 minutes more per run |
| **Confident switch** (runs 14–17) | Mirror of the confident-stay gate: if p(change) ≥ τ_high, exclude the current agent | Run 14: 70 → 71 (fixed 4, broke 3); run 16: 73 → 69; run 17: 72 → 69 | Below the +2 rule, then negative. It broke "aspect" and follow-up turns at p ≥ 0.99 |
| **Unified detector** (run 16) | Ask the routing model the yes / no context-change question instead of a separate detector | Gate 72 vs **73**; held-out 5,324 vs **5,733** of 5,948 | Weaker; the separate ModernBERT detector stayed |

**What survived:** the one-sided **confident-stay gate**. If the ModernBERT detector is sure the topic did *not* change (p < τ, chosen on dev), keep the current agent; otherwise let the router choose among **all** agents. It can only keep an agent, never exclude one, which is why it was the only combination that beat the router alone on unseen companies: **74 vs 68 / 83** (run 10).

### History and data

| Idea | What it did | Result | Why it was dropped (or kept) |
|---|---|---|---|
| **Windowing** (run 10b) | Router and detector read only the last N user turns | Router N = 3: 68 / 83, the same as full history; gate with windows 71 vs **74** | No gain. The decider gained +7 from a window, but it routes blind; we pass the current agent, and the gate already fixes the turns the window fixed for it |
| **Collapse to one statement** (run 9) | Rewrite the conversation as one sentence and route it as a first message | Fixed 1 of 10 router failures | Throws away order and recency; a one-off diagnostic |
| **Realistic training data** (run 11) | 744 free-form turns from 40 businesses | E10 test switches 15 → **12** / 20; gate 74 → 70 | **Kept** (more variety in phrasing) and followed with switch-heavy data (run 13): banking 41 → **47 / 50**, 188 → **205 / 225**, ding-dong 16 → 19 / 25, but E10 test stayed at 70 |

**The failure that remained:** on real switches *into a closely related agent*, the router stayed with the current agent ("could it be the chicken recipe?" stayed with Orders instead of Nutrition). The gate can't fix that, because it can only keep the current agent.

### General choice questions on a frozen encoder: the ceiling

The same input format answers grounded questions about a text (yes / no, choice, score), with the question in its own segment. The same 170 E12 items were given to the 2B Strands decider:

| Run | Setup | E12 (170) | yes / no (58) | choice (59) | score (53) |
|---|---|---|---|---|---|
| 14 | Router, zero-shot | 96 | 29 (chance) | 39 | 28 |
| 16 | One shared head, 1,524 choice items | 113 | 35 | 44 | 34 |
| 17 | Separate choice head, **9,113** items (6×) | **114** | 38 | 44 | 32 |
| — | Strands decider 2B | **160** | 56 | 56 | 48 |

**Six times the data gave +1.** The misses need *reading*: which detail the text states ("Which main did the customer order?" → soup instead of fish), comparing numbers ("Who scored highest?" → the lowest), and noticing that the same text gets a different question. Pooled vectors from a frozen encoder don't carry those answers, and no head on top can create them. **The encoder has to learn.** That's the next notebook.

### A finding about measurement

Runs 13 and 14 trained the same router on the same data with the same settings, yet banking moved from 47 · 205 to 43 · 198. The final head's epoch count was the median of the folds' best epochs, and one fold flipped between 8 and 2. **Run-to-run differences of a few banking conversations, or 1–2 E10 test turns, are noise.** From then on, results are compared against that noise band.

---
# Next

The best frozen system routes well: the hybrid (Nemotron router + ModernBERT confident-stay gate) scored 74 / 83 on routing in conversations at unseen companies, against 62 for the 2B decider with full history. But general questions stall at 114 / 170.

**`03_lora_signpost.ipynb`** (branch `signpost`) fine-tunes the encoder itself with LoRA: 7.2 M trained parameters on top of the 395 M frozen ones. With that, one model handles both routing and general questions, with no detector or gate. That model is **Signpost**, published at [huggingface.co/skamalj/signpost](https://huggingface.co/skamalj/signpost).